# HeartTrialTwin: Heart-Failure Clinical Trial Cohort

## Project Goal
HeartTrialTwin is a condition-specific clinical-trial intelligence project focused on heart failure.

The project will use historical clinical-trial data to:
- characterize the heart-failure treatment landscape,
- identify trial-design factors associated with termination, withdrawal, or unusually long completion time,
- retrieve comparable historical trials,
- and support trial-level what-if simulation.

## Initial Cohort
The initial cohort will include interventional studies related to heart failure.

Where supported by the data, trials will be categorized as:
- HFrEF — Heart Failure with Reduced Ejection Fraction
- HFmrEF — Heart Failure with Mildly Reduced Ejection Fraction
- HFpEF — Heart Failure with Preserved Ejection Fraction
- General or unspecified heart failure

## Important Modeling Rule
For prediction, only variables available at or near trial registration will be considered as candidate predictors.

Later-updated information, reported outcomes, reasons for stopping, and other post-registration information will not be used as prediction features.

## Current Notebook Objective
This notebook will:
1. Build the heart-failure trial cohort.
2. Explore trial statuses and dates.
3. Identify heart-failure subtypes.
4. Examine interventions and treatment classes.
5. Examine trial phases and trial-design characteristics.
6. Assess missingness and potential data leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
!pip install psycopg2-binary -q

In [ ]:
import psycopg2
from getpass import getpass

In [ ]:
AACT_USER = input("Enter your AACT username: ")
AACT_PASSWORD = getpass("Enter your AACT password: ")

conn = psycopg2.connect(
    host="aact-db.ctti-clinicaltrials.org",
    port=5432,
    dbname="aact",
    user=AACT_USER,
    password=AACT_PASSWORD,
    sslmode="require"
)

print("Connected to AACT successfully!")

In [ ]:
test_query = """
SELECT
    nct_id,
    brief_title,
    overall_status,
    study_type
FROM ctgov.studies
LIMIT 5;
"""

test_df = pd.read_sql_query(test_query, conn)

test_df

In [ ]:
!pip install sqlalchemy -q

In [ ]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

encoded_password = quote_plus(AACT_PASSWORD)

engine = create_engine(
    f"postgresql+psycopg2://{AACT_USER}:{encoded_password}"
    "@aact-db.ctti-clinicaltrials.org:5432/aact"
)

In [ ]:
test_df = pd.read_sql_query(test_query, engine)

test_df

In [ ]:
from sqlalchemy import text

heart_failure_query = text("""
SELECT DISTINCT
    s.nct_id,
    s.brief_title,
    s.overall_status,
    s.phase,
    s.enrollment,
    s.start_date,
    s.completion_date,
    c.name AS condition_name,
    bc.mesh_term
FROM ctgov.studies s
LEFT JOIN ctgov.conditions c
    ON s.nct_id = c.nct_id
LEFT JOIN ctgov.browse_conditions bc
    ON s.nct_id = bc.nct_id
WHERE s.study_type = 'INTERVENTIONAL'
  AND (
       LOWER(c.name) LIKE :hf_term
       OR LOWER(bc.mesh_term) LIKE :hf_term
  );
""")

hf_candidates = pd.read_sql_query(
    heart_failure_query,
    engine,
    params={"hf_term": "%heart failure%"}
)

hf_candidates.head()

In [ ]:
print("Rows returned:", len(hf_candidates))
print("Unique trials:", hf_candidates["nct_id"].nunique())

In [ ]:
hf_candidates[
    ["nct_id", "condition_name", "mesh_term"]
].head(20)

In [ ]:
hf_one_row = (
    hf_candidates
    .groupby(
        [
            "nct_id",
            "brief_title",
            "overall_status",
            "phase",
            "enrollment",
            "start_date",
            "completion_date"
        ],
        dropna=False
    )
    .agg(
        condition_names=("condition_name", lambda x: sorted(set(x.dropna()))),
        mesh_terms=("mesh_term", lambda x: sorted(set(x.dropna())))
    )
    .reset_index()
)

print("Rows after collapsing:", len(hf_one_row))
print("Unique trials:", hf_one_row["nct_id"].nunique())

hf_one_row.head()

In [ ]:
status_counts = (
    hf_one_row["overall_status"]
    .value_counts(dropna=False)
    .reset_index()
)

status_counts.columns = ["overall_status", "count"]

status_counts["percentage"] = (
    status_counts["count"] / len(hf_one_row) * 100
).round(2)

status_counts

In [ ]:
phase_counts = (
    hf_one_row["phase"]
    .value_counts(dropna=False)
    .reset_index()
)

phase_counts.columns = ["phase", "count"]

phase_counts["percentage"] = (
    phase_counts["count"] / len(hf_one_row) * 100
).round(2)

phase_counts

In [ ]:
status_by_phase = pd.crosstab(
    hf_one_row["phase"],
    hf_one_row["overall_status"],
    margins=True
)

status_by_phase

In [ ]:
status_by_phase_pct = pd.crosstab(
    hf_one_row["phase"],
    hf_one_row["overall_status"],
    normalize="index"
) * 100

status_by_phase_pct = status_by_phase_pct.round(1)

status_by_phase_pct

In [ ]:
hf_ids = tuple(hf_one_row["nct_id"].tolist())

intervention_query = text("""
SELECT
    i.nct_id,
    i.intervention_type,
    i.name AS intervention_name,
    i.description
FROM ctgov.interventions i
WHERE i.nct_id IN :nct_ids
ORDER BY i.nct_id;
""").bindparams(
    __import__("sqlalchemy").bindparam("nct_ids", expanding=True)
)

hf_interventions = pd.read_sql_query(
    intervention_query,
    engine,
    params={"nct_ids": hf_ids}
)

print("Intervention records:", len(hf_interventions))
print("Trials with intervention records:", hf_interventions["nct_id"].nunique())

hf_interventions.head(20)

In [ ]:
intervention_type_counts = (
    hf_interventions["intervention_type"]
    .value_counts(dropna=False)
    .reset_index()
)

intervention_type_counts.columns = ["intervention_type", "count"]

intervention_type_counts

In [ ]:
intervention_trial_counts = (
    hf_interventions
    .groupby("intervention_type")["nct_id"]
    .nunique()
    .reset_index(name="trial_count")
    .sort_values("trial_count", ascending=False)
)

intervention_trial_counts["percentage_of_trials"] = (
    intervention_trial_counts["trial_count"]
    / hf_one_row["nct_id"].nunique()
    * 100
).round(2)

intervention_trial_counts

In [ ]:
drug_indicator = (
    hf_interventions
    .assign(
        has_drug=hf_interventions["intervention_type"].eq("DRUG")
    )
    .groupby("nct_id")["has_drug"]
    .max()
    .reset_index()
)

hf_phase_check = hf_one_row.merge(
    drug_indicator,
    on="nct_id",
    how="left"
)

hf_phase_check["has_drug"] = (
    hf_phase_check["has_drug"]
    .fillna(False)
    .astype(bool)
)

phase_by_drug = pd.crosstab(
    hf_phase_check["has_drug"],
    hf_phase_check["phase"],
    margins=True
)

phase_by_drug

In [ ]:
summary_query = text("""
SELECT
    bs.nct_id,
    bs.description AS brief_summary
FROM ctgov.brief_summaries bs
WHERE bs.nct_id IN :nct_ids;
""").bindparams(
    __import__("sqlalchemy").bindparam("nct_ids", expanding=True)
)

hf_summaries = pd.read_sql_query(
    summary_query,
    engine,
    params={"nct_ids": hf_ids}
)

print("Summary records:", len(hf_summaries))
print("Trials with summaries:", hf_summaries["nct_id"].nunique())

hf_summaries.head()

In [ ]:
hf_cohort = hf_one_row.merge(
    hf_summaries,
    on="nct_id",
    how="left"
)

print("Rows:", len(hf_cohort))
print("Unique trials:", hf_cohort["nct_id"].nunique())
print("Missing summaries:", hf_cohort["brief_summary"].isna().sum())

hf_cohort.head()

In [ ]:
hf_cohort["hf_text"] = (
    hf_cohort["brief_title"].fillna("").astype(str)
    + " "
    + hf_cohort["condition_names"].apply(
        lambda x: " ".join(x) if isinstance(x, list) else ""
    )
    + " "
    + hf_cohort["mesh_terms"].apply(
        lambda x: " ".join(x) if isinstance(x, list) else ""
    )
    + " "
    + hf_cohort["brief_summary"].fillna("").astype(str)
).str.lower()

hf_cohort[
    ["nct_id", "brief_title", "hf_text"]
].head()

In [ ]:
subtype_terms = {
    "HFpEF": [
        "hfpef",
        "heart failure with preserved ejection fraction",
        "preserved ejection fraction"
    ],

    "HFrEF": [
        "hfref",
        "heart failure with reduced ejection fraction",
        "reduced ejection fraction"
    ],

    "HFmrEF": [
        "hfmref",
        "heart failure with mildly reduced ejection fraction",
        "heart failure with mid-range ejection fraction",
        "mid-range ejection fraction",
        "mildly reduced ejection fraction"
    ]
}

for subtype, terms in subtype_terms.items():
    pattern = "|".join(terms)
    count = hf_cohort["hf_text"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    ).sum()

    print(f"{subtype}: {count}")

In [ ]:
hf_cohort["mentions_hfpef"] = hf_cohort["hf_text"].str.contains(
    r"\bhfpef\b|heart failure with preserved ejection fraction|preserved ejection fraction",
    case=False,
    na=False,
    regex=True
)

hf_cohort["mentions_hfref"] = hf_cohort["hf_text"].str.contains(
    r"\bhfref\b|heart failure with reduced ejection fraction|reduced ejection fraction",
    case=False,
    na=False,
    regex=True
)

hf_cohort["mentions_hfmref"] = hf_cohort["hf_text"].str.contains(
    r"\bhfmref\b|heart failure with mildly reduced ejection fraction|heart failure with mid-range ejection fraction|mildly reduced ejection fraction|mid-range ejection fraction",
    case=False,
    na=False,
    regex=True
)

In [ ]:
subtype_overlap = (
    hf_cohort
    .groupby(
        ["mentions_hfpef", "mentions_hfref", "mentions_hfmref"]
    )
    .size()
    .reset_index(name="trial_count")
    .sort_values("trial_count", ascending=False)
)

subtype_overlap

In [ ]:
hf_cohort.loc[
    hf_cohort[
        ["mentions_hfpef", "mentions_hfref", "mentions_hfmref"]
    ].any(axis=1),
    [
        "nct_id",
        "brief_title",
        "mentions_hfpef",
        "mentions_hfref",
        "mentions_hfmref"
    ]
].head(30)

In [ ]:
strong_subtype_patterns = {
    "HFpEF": (
        r"\bhfpef\b|"
        r"heart failure with preserved ejection fraction|"
        r"heart failure with preserved ef|"
        r"diastolic heart failure"
    ),

    "HFrEF": (
        r"\bhfref\b|"
        r"heart failure with reduced ejection fraction|"
        r"heart failure with reduced ef|"
        r"systolic heart failure"
    ),

    "HFmrEF": (
        r"\bhfmref\b|"
        r"heart failure with mildly reduced ejection fraction|"
        r"heart failure with mid-range ejection fraction|"
        r"heart failure with mildly reduced ef|"
        r"heart failure with mid-range ef"
    )
}

for subtype, pattern in strong_subtype_patterns.items():
    hf_cohort[f"strong_{subtype.lower()}"] = (
        hf_cohort["hf_text"]
        .str.contains(pattern, case=False, na=False, regex=True)
    )

In [ ]:
strong_subtype_counts = pd.DataFrame({
    "HF_subtype": ["HFpEF", "HFrEF", "HFmrEF"],
    "trial_count": [
        hf_cohort["strong_hfpef"].sum(),
        hf_cohort["strong_hfref"].sum(),
        hf_cohort["strong_hfmref"].sum()
    ]
})

strong_subtype_counts

In [ ]:
strong_subtype_overlap = (
    hf_cohort
    .groupby(
        ["strong_hfpef", "strong_hfref", "strong_hfmref"]
    )
    .size()
    .reset_index(name="trial_count")
    .sort_values("trial_count", ascending=False)
)

strong_subtype_overlap

In [ ]:
def assign_hf_subtype(row):
    subtypes = []

    if row["strong_hfpef"]:
        subtypes.append("HFpEF")

    if row["strong_hfref"]:
        subtypes.append("HFrEF")

    if row["strong_hfmref"]:
        subtypes.append("HFmrEF")

    if len(subtypes) == 0:
        return "General/Unspecified HF"
    elif len(subtypes) == 1:
        return subtypes[0]
    else:
        return "Mixed/Multiple"


hf_cohort["hf_subtype"] = hf_cohort.apply(
    assign_hf_subtype,
    axis=1
)

In [ ]:
subtype_distribution = (
    hf_cohort["hf_subtype"]
    .value_counts()
    .reset_index()
)

subtype_distribution.columns = [
    "hf_subtype",
    "trial_count"
]

subtype_distribution["percentage"] = (
    subtype_distribution["trial_count"]
    / len(hf_cohort)
    * 100
).round(2)

subtype_distribution

In [ ]:
hf_cohort["hf_in_title"] = (
    hf_cohort["brief_title"]
    .fillna("")
    .str.contains(
        r"heart failure|cardiac failure|ventricular dysfunction",
        case=False,
        regex=True
    )
)

hf_cohort["hf_in_condition"] = (
    hf_cohort["condition_names"]
    .apply(
        lambda conditions: any(
            "heart failure" in str(condition).lower()
            for condition in conditions
        )
        if isinstance(conditions, list)
        else False
    )
)

hf_cohort["hf_in_mesh"] = (
    hf_cohort["mesh_terms"]
    .apply(
        lambda terms: any(
            "heart failure" in str(term).lower()
            for term in terms
        )
        if isinstance(terms, list)
        else False
    )
)

hf_cohort["hf_in_summary"] = (
    hf_cohort["brief_summary"]
    .fillna("")
    .str.contains(
        r"heart failure|cardiac failure|ventricular dysfunction",
        case=False,
        regex=True
    )
)

hf_evidence_counts = pd.DataFrame({
    "evidence_source": [
        "Title",
        "Submitted condition",
        "MeSH condition",
        "Brief summary"
    ],
    "trial_count": [
        hf_cohort["hf_in_title"].sum(),
        hf_cohort["hf_in_condition"].sum(),
        hf_cohort["hf_in_mesh"].sum(),
        hf_cohort["hf_in_summary"].sum()
    ]
})

hf_evidence_counts["percentage"] = (
    hf_evidence_counts["trial_count"]
    / len(hf_cohort)
    * 100
).round(2)

hf_evidence_counts

In [ ]:
evidence_cols = [
    "hf_in_title",
    "hf_in_condition",
    "hf_in_mesh",
    "hf_in_summary"
]

hf_cohort["hf_evidence_score"] = (
    hf_cohort[evidence_cols]
    .sum(axis=1)
)

evidence_score_distribution = (
    hf_cohort["hf_evidence_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)

evidence_score_distribution.columns = [
    "number_of_evidence_sources",
    "trial_count"
]

evidence_score_distribution["percentage"] = (
    evidence_score_distribution["trial_count"]
    / len(hf_cohort)
    * 100
).round(2)

evidence_score_distribution

In [ ]:
single_evidence = hf_cohort[
    hf_cohort["hf_evidence_score"] == 1
].copy()

single_evidence["only_evidence_source"] = np.select(
    [
        single_evidence["hf_in_title"],
        single_evidence["hf_in_condition"],
        single_evidence["hf_in_mesh"],
        single_evidence["hf_in_summary"]
    ],
    [
        "Title only",
        "Submitted condition only",
        "MeSH only",
        "Brief summary only"
    ],
    default="Unknown"
)

single_evidence_distribution = (
    single_evidence["only_evidence_source"]
    .value_counts()
    .reset_index()
)

single_evidence_distribution.columns = [
    "only_evidence_source",
    "trial_count"
]

single_evidence_distribution["percentage_of_single_source"] = (
    single_evidence_distribution["trial_count"]
    / len(single_evidence)
    * 100
).round(2)

single_evidence_distribution

In [ ]:
mesh_only_trials = single_evidence[
    single_evidence["only_evidence_source"] == "MeSH only"
].copy()

print("Number of MeSH-only trials:", len(mesh_only_trials))

mesh_only_trials[
    [
        "nct_id",
        "brief_title",
        "condition_names",
        "mesh_terms",
        "brief_summary"
    ]
].head(30)

In [ ]:
hf_condition_patterns = [
    "heart failure",
    "cardiac failure",
    "left ventricular failure",
    "right ventricular failure",
    "ventricular failure",
    "congestive cardiac failure"
]

hf_cohort["hf_in_condition_expanded"] = (
    hf_cohort["condition_names"]
    .apply(
        lambda conditions: any(
            any(
                term in str(condition).lower()
                for term in hf_condition_patterns
            )
            for condition in conditions
        )
        if isinstance(conditions, list)
        else False
    )
)

print(
    "Original submitted-condition matches:",
    hf_cohort["hf_in_condition"].sum()
)

print(
    "Expanded submitted-condition matches:",
    hf_cohort["hf_in_condition_expanded"].sum()
)

print(
    "Additional trials identified:",
    (
        hf_cohort["hf_in_condition_expanded"]
        & ~hf_cohort["hf_in_condition"]
    ).sum()
)

In [ ]:
mesh_only_trials = hf_cohort[
    (hf_cohort["hf_in_mesh"] == True) &
    (hf_cohort["hf_in_condition"] == False) &
    (hf_cohort["hf_in_title"] == False) &
    (hf_cohort["hf_in_summary"] == False)
].copy()

mesh_only_trials["rescued_by_expanded_condition"] = (
    mesh_only_trials["hf_in_condition_expanded"]
)

rescue_counts = (
    mesh_only_trials["rescued_by_expanded_condition"]
    .value_counts()
    .rename(index={
        True: "Now recognized by expanded condition",
        False: "Still MeSH only"
    })
    .reset_index()
)

rescue_counts.columns = ["category", "trial_count"]

rescue_counts

In [ ]:
true_mesh_only = mesh_only_trials[
    mesh_only_trials["rescued_by_expanded_condition"] == False
].copy()

mesh_only_conditions = (
    true_mesh_only[["nct_id", "condition_names"]]
    .explode("condition_names")
    .groupby("condition_names")
    .agg(
        trial_count=("nct_id", "nunique")
    )
    .sort_values("trial_count", ascending=False)
    .reset_index()
)

print("Remaining true MeSH-only trials:", len(true_mesh_only))

mesh_only_conditions.head(30)

In [ ]:
# Explicit HF evidence in the title or summary
hf_cohort["hf_in_text_expanded"] = (
    hf_cohort["brief_title"].fillna("")
    + " "
    + hf_cohort["brief_summary"].fillna("")
).str.contains(
    r"heart failure|"
    r"cardiac failure|"
    r"left ventricular failure|"
    r"right ventricular failure|"
    r"ventricular failure|"
    r"congestive cardiac failure",
    case=False,
    na=False,
    regex=True
)

# Primary inclusion rule:
# explicit HF/failure terminology in submitted condition,
# title, or brief summary
hf_cohort["primary_hf_inclusion"] = (
    hf_cohort["hf_in_condition_expanded"]
    | hf_cohort["hf_in_text_expanded"]
)

primary_hf_cohort = (
    hf_cohort[
        hf_cohort["primary_hf_inclusion"]
    ]
    .copy()
    .reset_index(drop=True)
)

excluded_from_primary_hf = (
    hf_cohort[
        ~hf_cohort["primary_hf_inclusion"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Original candidate trials:", len(hf_cohort))
print("Primary HF cohort:", len(primary_hf_cohort))
print("Excluded from primary cohort:", len(excluded_from_primary_hf))
print(
    "Primary cohort retained:",
    round(len(primary_hf_cohort) / len(hf_cohort) * 100, 2),
    "%"
)

In [ ]:
primary_hf_ids = set(primary_hf_cohort["nct_id"])

primary_hf_interventions = (
    hf_interventions[
        hf_interventions["nct_id"].isin(primary_hf_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Primary HF trials:", len(primary_hf_cohort))
print("Intervention records:", len(primary_hf_interventions))
print(
    "Trials with intervention records:",
    primary_hf_interventions["nct_id"].nunique()
)

intervention_type_summary = (
    primary_hf_interventions
    .groupby("intervention_type")
    .agg(
        intervention_records=("nct_id", "size"),
        unique_trials=("nct_id", "nunique")
    )
    .sort_values("unique_trials", ascending=False)
    .reset_index()
)

intervention_type_summary["percentage_of_hf_trials"] = (
    intervention_type_summary["unique_trials"]
    / len(primary_hf_cohort)
    * 100
).round(2)

intervention_type_summary

In [ ]:
hf_drugs = (
    primary_hf_interventions[
        primary_hf_interventions["intervention_type"] == "DRUG"
    ]
    .copy()
)

drug_name_summary = (
    hf_drugs
    .groupby("intervention_name")
    .agg(
        intervention_records=("nct_id", "size"),
        unique_trials=("nct_id", "nunique")
    )
    .sort_values(
        ["unique_trials", "intervention_records"],
        ascending=False
    )
    .reset_index()
)

print("Drug intervention records:", len(hf_drugs))
print("HF trials containing a drug:", hf_drugs["nct_id"].nunique())
print("Unique recorded drug names:", hf_drugs["intervention_name"].nunique())

drug_name_summary.head(50)

In [ ]:
import re

def normalize_drug_name(name):
    if pd.isna(name):
        return np.nan

    name = str(name).lower().strip()

    # Standardize separators
    name = name.replace("/", " ")
    name = name.replace("-", " ")

    # Remove common dosage expressions
    name = re.sub(
        r"\b\d+(\.\d+)?\s*(mg|mcg|µg|ug|g|ml|iu|units?)\b",
        " ",
        name
    )

    # Remove common formulation words
    name = re.sub(
        r"\b(tablet|tablets|tab|tabs|capsule|capsules|"
        r"injection|injectable|oral|solution|infusion)\b",
        " ",
        name
    )

    # Remove punctuation except letters/numbers/spaces
    name = re.sub(r"[^a-z0-9\s]", " ", name)

    # Collapse repeated spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name


hf_drugs["drug_name_normalized"] = (
    hf_drugs["intervention_name"]
    .apply(normalize_drug_name)
)

normalized_drug_summary = (
    hf_drugs
    .groupby("drug_name_normalized")
    .agg(
        unique_trials=("nct_id", "nunique"),
        original_names=(
            "intervention_name",
            lambda x: sorted(set(x))
        )
    )
    .sort_values("unique_trials", ascending=False)
    .reset_index()
)

print(
    "Original unique drug names:",
    hf_drugs["intervention_name"].nunique()
)

print(
    "Normalized unique drug names:",
    hf_drugs["drug_name_normalized"].nunique()
)

normalized_drug_summary.head(50)

In [ ]:
def classify_hf_drug(name):
    if pd.isna(name):
        return "Other/Unclassified Drug"

    name = name.lower().strip()

    # Comparator / control
    if (
        "placebo" in name
        or name == "saline"
        or "standard of care" in name
        or name == "gdmt"
    ):
        return "Comparator/Background Therapy"

    # SGLT inhibitors
    if any(drug in name for drug in [
        "dapagliflozin",
        "empagliflozin",
        "sotagliflozin",
        "ertugliflozin",
        "canagliflozin"
    ]):
        return "SGLT Inhibitor"

    # ARNI
    if (
        "sacubitril valsartan" in name
        or "lcz696" in name
    ):
        return "ARNI"

    # Mineralocorticoid receptor antagonists
    if any(drug in name for drug in [
        "spironolactone",
        "eplerenone",
        "finerenone"
    ]):
        return "Mineralocorticoid Receptor Antagonist"

    # Beta blockers
    if any(drug in name for drug in [
        "carvedilol",
        "bisoprolol",
        "metoprolol",
        "nebivolol"
    ]):
        return "Beta Blocker"

    # ACE inhibitors
    if any(drug in name for drug in [
        "enalapril",
        "lisinopril",
        "ramipril",
        "captopril",
        "perindopril"
    ]):
        return "ACE Inhibitor"

    # Angiotensin receptor blockers
    if any(drug in name for drug in [
        "valsartan",
        "candesartan",
        "losartan",
        "irbesartan",
        "telmisartan"
    ]):
        return "ARB"

    # Diuretics
    if any(drug in name for drug in [
        "furosemide",
        "bumetanide",
        "torsemide",
        "metolazone",
        "acetazolamide"
    ]):
        return "Diuretic"

    # Selected other HF-relevant classes
    if "ivabradine" in name:
        return "If Channel Inhibitor"

    if "digoxin" in name:
        return "Cardiac Glycoside"

    if "vericiguat" in name:
        return "sGC Stimulator"

    if "levosimendan" in name:
        return "Calcium Sensitizer/Inodilator"

    return "Other/Unclassified Drug"


hf_drugs["treatment_class"] = (
    hf_drugs["drug_name_normalized"]
    .apply(classify_hf_drug)
)

treatment_class_summary = (
    hf_drugs
    .groupby("treatment_class")
    .agg(
        unique_trials=("nct_id", "nunique"),
        intervention_records=("nct_id", "size")
    )
    .sort_values("unique_trials", ascending=False)
    .reset_index()
)

treatment_class_summary["percentage_of_drug_trials"] = (
    treatment_class_summary["unique_trials"]
    / hf_drugs["nct_id"].nunique()
    * 100
).round(2)

treatment_class_summary

In [ ]:
unclassified_drugs = (
    hf_drugs[
        hf_drugs["treatment_class"] == "Other/Unclassified Drug"
    ]
    .groupby("drug_name_normalized")
    .agg(
        unique_trials=("nct_id", "nunique"),
        intervention_records=("nct_id", "size")
    )
    .sort_values(
        ["unique_trials", "intervention_records"],
        ascending=False
    )
    .reset_index()
)

print(
    "Unique normalized names still unclassified:",
    unclassified_drugs["drug_name_normalized"].nunique()
)

print(
    "Trials containing at least one unclassified drug:",
    hf_drugs.loc[
        hf_drugs["treatment_class"] == "Other/Unclassified Drug",
        "nct_id"
    ].nunique()
)

unclassified_drugs.head(50)

In [ ]:
def refine_treatment_class(row):
    name = row["drug_name_normalized"]
    current_class = row["treatment_class"]

    # Keep anything already classified
    if current_class != "Other/Unclassified Drug":
        return current_class

    if pd.isna(name):
        return current_class

    name = name.lower().strip()

    # Additional comparator/background therapy terms
    if any(term in name for term in [
        "optimal medical therapy",
        "guideline directed medical therapy"
    ]):
        return "Comparator/Background Therapy"

    # Generic versions / alternative names of existing classes
    if "sglt2 inhibitor" in name:
        return "SGLT Inhibitor"

    if "beta blocker" in name:
        return "Beta Blocker"

    if "loop diuretic" in name or name == "diuretics":
        return "Diuretic"

    if any(term in name for term in [
        "entresto",
        "lcz 696"
    ]):
        return "ARNI"

    # Vasopressin receptor antagonists
    if any(drug in name for drug in [
        "tolvaptan",
        "conivaptan"
    ]):
        return "Vasopressin Receptor Antagonist"

    # Natriuretic peptide therapies
    if any(drug in name for drug in [
        "nesiritide",
        "cd np"
    ]):
        return "Natriuretic Peptide Therapy"

    # PDE-5 inhibitor
    if "sildenafil" in name:
        return "PDE-5 Inhibitor"

    # Intravenous iron
    if any(term in name for term in [
        "ferric carboxymaltose",
        "ferric derisomaltose",
        "ferinject"
    ]):
        return "Intravenous Iron"

    # Inotropes / inodilators
    if any(drug in name for drug in [
        "dobutamine",
        "milrinone",
        "istaroxime"
    ]):
        return "Inotrope/Inodilator"

    # Cardiac myosin activator
    if any(term in name for term in [
        "omecamtiv mecarbil",
        "ck 1827452"
    ]):
        return "Cardiac Myosin Activator"

    return current_class


hf_drugs["treatment_class_refined"] = hf_drugs.apply(
    refine_treatment_class,
    axis=1
)

refined_class_summary = (
    hf_drugs
    .groupby("treatment_class_refined")
    .agg(
        unique_trials=("nct_id", "nunique"),
        intervention_records=("nct_id", "size")
    )
    .sort_values("unique_trials", ascending=False)
    .reset_index()
)

refined_class_summary["percentage_of_drug_trials"] = (
    refined_class_summary["unique_trials"]
    / hf_drugs["nct_id"].nunique()
    * 100
).round(2)

refined_class_summary

In [ ]:
# One record per trial-treatment class
trial_treatment_classes = (
    hf_drugs[
        ["nct_id", "treatment_class_refined"]
    ]
    .drop_duplicates()
)

# Add HF subtype information
trial_treatment_subtype = (
    trial_treatment_classes
    .merge(
        primary_hf_cohort[
            ["nct_id", "hf_subtype"]
        ],
        on="nct_id",
        how="left"
    )
)

# Count unique trials for each treatment class × HF subtype
treatment_by_subtype = (
    trial_treatment_subtype
    .groupby(
        ["treatment_class_refined", "hf_subtype"]
    )
    ["nct_id"]
    .nunique()
    .unstack(fill_value=0)
)

# Add total number of unique trials represented by each treatment class
treatment_by_subtype["Total"] = (
    treatment_by_subtype.sum(axis=1)
)

treatment_by_subtype = (
    treatment_by_subtype
    .sort_values("Total", ascending=False)
)

treatment_by_subtype

In [ ]:
# Add phase information to the unique trial-treatment class table
trial_treatment_phase = (
    trial_treatment_classes
    .merge(
        primary_hf_cohort[
            ["nct_id", "phase"]
        ],
        on="nct_id",
        how="left"
    )
)
# Make missing or non-applicable phase explicit
trial_treatment_phase["phase_display"] = (
    trial_treatment_phase["phase"]
    .replace("NA", "NA / Not Applicable or Not Reported")
    .fillna("NA / Not Applicable or Not Reported")
)

# Count unique trials for each treatment class × phase
treatment_by_phase = (
    trial_treatment_phase
    .groupby(
        ["treatment_class_refined", "phase_display"]
    )["nct_id"]
    .nunique()
    .unstack(fill_value=0)
)

# Add total trials represented by each treatment class
treatment_by_phase["Total"] = (
    treatment_by_phase.sum(axis=1)
)

treatment_by_phase = (
    treatment_by_phase
    .sort_values("Total", ascending=False)
)

treatment_by_phase

In [ ]:
# Prepare treatment-class landscape figure
plot_treatment_classes = (
    refined_class_summary[
        ~refined_class_summary["treatment_class_refined"].isin(
            [
                "Other/Unclassified Drug",
                "Comparator/Background Therapy"
            ]
        )
    ]
    .sort_values("unique_trials", ascending=True)
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_treatment_classes["treatment_class_refined"],
    plot_treatment_classes["unique_trials"]
)

plt.xlabel("Number of Unique Heart-Failure Trials")
plt.ylabel("Treatment Class")
plt.title("Heart-Failure Drug Treatment Landscape")

plt.tight_layout()
plt.show()

In [ ]:
# Select major treatment classes for subtype comparison
major_classes = [
    "SGLT Inhibitor",
    "Diuretic",
    "ARNI",
    "Mineralocorticoid Receptor Antagonist",
    "Beta Blocker",
    "ARB",
    "ACE Inhibitor",
    "Natriuretic Peptide Therapy",
    "Calcium Sensitizer/Inodilator",
    "Vasopressin Receptor Antagonist"
]

subtype_plot_data = (
    treatment_by_subtype
    .loc[
        treatment_by_subtype.index.isin(major_classes),
        ["General/Unspecified HF", "HFpEF", "HFrEF"]
    ]
)

# Order by total trials across the displayed subtypes
subtype_plot_data["Displayed Total"] = subtype_plot_data.sum(axis=1)

subtype_plot_data = (
    subtype_plot_data
    .sort_values("Displayed Total", ascending=True)
    .drop(columns="Displayed Total")
)

subtype_plot_data.plot(
    kind="barh",
    figsize=(11, 7)
)

plt.xlabel("Number of Unique Heart-Failure Trials")
plt.ylabel("Treatment Class")
plt.title("Heart-Failure Treatment Classes by HF Subtype")
plt.legend(
    title="HF Subtype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
# Save the main Week 2 datasets

primary_hf_cohort.to_csv(
    "HeartTrialTwin_primary_HF_cohort.csv",
    index=False
)

excluded_from_primary_hf.to_csv(
    "HeartTrialTwin_excluded_HF_candidates.csv",
    index=False
)

primary_hf_interventions.to_csv(
    "HeartTrialTwin_primary_HF_interventions.csv",
    index=False
)

hf_drugs.to_csv(
    "HeartTrialTwin_HF_drug_classes.csv",
    index=False
)

treatment_by_subtype.to_csv(
    "HeartTrialTwin_treatment_by_HF_subtype.csv"
)

treatment_by_phase.to_csv(
    "HeartTrialTwin_treatment_by_phase.csv"
)

print("Week 2 datasets saved successfully.")
print()
print("Files created:")
print("1. HeartTrialTwin_primary_HF_cohort.csv")
print("2. HeartTrialTwin_excluded_HF_candidates.csv")
print("3. HeartTrialTwin_primary_HF_interventions.csv")
print("4. HeartTrialTwin_HF_drug_classes.csv")
print("5. HeartTrialTwin_treatment_by_HF_subtype.csv")
print("6. HeartTrialTwin_treatment_by_phase.csv")

## Week 2 Progress Summary

### 1. Heart-Failure Cohort Construction
- Connected to the AACT clinical-trial database and retrieved interventional studies related to heart failure.
- Initial broad retrieval identified 4,704 candidate trials.
- Candidate trials were identified using submitted condition terminology and NLM-generated MeSH terminology to maximize retrieval sensitivity.
- For the primary analytical cohort, trials were required to contain explicit heart-failure or equivalent failure terminology in the submitted condition, title, or brief summary.
- Trials identified only through MeSH mapping without explicit supporting heart-failure terminology were excluded from the primary cohort.
- Final primary heart-failure cohort: **4,554 trials (96.81% of candidates).**

### 2. Initial Exploratory Analysis
The primary cohort was explored according to:
- overall trial status,
- trial phase,
- status by phase,
- intervention type,
- missingness and data-quality characteristics.

The analysis showed substantial heterogeneity in trial design and intervention type. Drug interventions were the largest individual intervention category, but device, behavioral, procedural, and other interventions were also well represented.

### 3. Heart-Failure Subtype Identification
A conservative text-based classification was developed using trial titles, submitted conditions, MeSH terminology, and brief summaries.

Trials were categorized as:
- HFrEF,
- HFpEF,
- HFmrEF,
- Mixed/Multiple,
- General/Unspecified HF.

Most historical trials did not explicitly identify one of the major HF phenotypes and therefore remained classified as General/Unspecified HF. HFmrEF had very limited explicit representation and will not be treated as a standalone modeling subgroup unless later data refinement provides sufficient observations.

### 4. Treatment-Landscape Development
The primary cohort contained:
- **7,731 intervention records**
- **3,011 drug intervention records**
- **1,667 trials containing at least one drug intervention**

Drug names were normalized to reduce differences caused by capitalization, dosage notation, formulation, and formatting.

Unique recorded drug names decreased from:
**1,594 raw names → 1,324 normalized names.**

A transparent rule-based treatment taxonomy was then developed for major HF-relevant therapeutic classes, including:
- SGLT inhibitors,
- ARNI,
- mineralocorticoid receptor antagonists,
- beta blockers,
- ACE inhibitors,
- ARBs,
- diuretics,
- natriuretic peptide therapies,
- vasopressin receptor antagonists,
- intravenous iron,
- inotropes/inodilators,
- cardiac myosin activators,
- and selected additional HF treatment classes.

Comparator and background therapies were separated from therapeutic treatment classes. Drugs that could not be reliably assigned were retained as Other/Unclassified rather than forcing potentially incorrect classifications.

### 5. Treatment-Landscape Findings
Among explicitly classified therapeutic categories, SGLT inhibitors were the most frequently represented, followed by diuretics, ARNI, mineralocorticoid receptor antagonists, and beta blockers.

Treatment classes were also compared across:
- HF subtype, and
- clinical trial phase.

The subtype analysis showed that a large proportion of historical trials remained General/Unspecified HF. Among explicitly classified subtypes, treatment-class representation varied across HFpEF and HFrEF.

These findings describe the historical clinical-trial research landscape and should not be interpreted as evidence of treatment effectiveness or causal treatment effects.

### 6. Week 2 Visualizations
Two initial treatment-landscape figures were produced:
1. Heart-Failure Drug Treatment Landscape.
2. Heart-Failure Treatment Classes by HF Subtype.

### 7. Important Limitations Identified
- Treatment classes are not intended to represent a complete pharmacological ontology.
- A substantial number of drugs remain Other/Unclassified.
- A single trial may contain multiple interventions and treatment classes.
- HF subtype information is frequently not explicitly reported in historical registry text.
- Current AACT records represent the current version of trial registrations and may contain fields updated after initial registration.
- Future predictive modeling must therefore carefully distinguish prediction-time variables from information that could introduce temporal data leakage.

## Week 2 Checkpoint

Week 2 established the condition-specific heart-failure cohort, performed initial exploratory analysis, developed conservative HF subtype features, and constructed the first treatment-landscape representation.

**No predictive outcome has been finalized and no risk model has been trained at this stage.**

## Week 3 Starting Point

Week 3 will focus on constructing the leakage-aware modeling dataset by retrieving and evaluating additional trial-level features such as:

- sponsor characteristics,
- enrollment,
- allocation and masking,
- intervention model,
- primary purpose,
- number of sites and countries,
- eligibility information,
- endpoint characteristics,
- trial start year,
- and potential measures of trial complexity.

Outcome and date distributions will then be evaluated before deciding whether the primary prediction task should focus on termination/withdrawal, unusually long completion time, or a combination of these outcomes.

## Final Cohort Outcome and Timeline Assessment

Now that the primary heart-failure cohort has been defined, this section evaluates trial status and date availability within the final cohort.

The purpose is to determine which outcome definitions are sufficiently supported for later trial-risk forecasting.

Potential outcomes include:
- trial termination or withdrawal,
- completed versus unsuccessful trial status,
- and unusually long trial completion time.

No modeling target is assigned at this stage. Outcome feasibility will first be assessed based on class distribution, ongoing studies, unknown statuses, and date completeness.

In [ ]:
# Examine status distribution in the final primary HF cohort

final_status_summary = (
    primary_hf_cohort["overall_status"]
    .value_counts(dropna=False)
    .reset_index()
)

final_status_summary.columns = [
    "overall_status",
    "trial_count"
]

final_status_summary["percentage"] = (
    final_status_summary["trial_count"]
    / len(primary_hf_cohort)
    * 100
).round(2)

print(
    "Total trials in primary HF cohort:",
    len(primary_hf_cohort)
)

final_status_summary

In [ ]:
# Check the current primary HF cohort and exclusion counts

print("Candidate rows:", len(hf_cohort))
print("Candidate unique trials:", hf_cohort["nct_id"].nunique())

print()

print("Primary HF cohort rows:", len(primary_hf_cohort))
print(
    "Primary HF cohort unique trials:",
    primary_hf_cohort["nct_id"].nunique()
)

print()

print(
    "Excluded trials:",
    (~hf_cohort["primary_hf_inclusion"]).sum()
)

### Cohort Version Note

The AACT database is updated over time as ClinicalTrials.gov records are added or modified. During development, a refreshed query returned 4,711 candidate heart-failure trials rather than the 4,704 trials observed in an earlier extraction.

Applying the same predefined primary inclusion criteria resulted in a final cohort of 4,561 trials, with 150 candidate trials excluded.

The 4,561-trial cohort is used as the fixed analytical cohort for subsequent analyses in this notebook. Saved cohort files are used for downstream analyses to maintain reproducibility and avoid changes caused by later database updates.

In [ ]:
# Assess date availability in the final primary HF cohort

date_availability = pd.DataFrame({
    "variable": [
        "start_date",
        "completion_date"
    ],
    "available_trials": [
        primary_hf_cohort["start_date"].notna().sum(),
        primary_hf_cohort["completion_date"].notna().sum()
    ],
    "missing_trials": [
        primary_hf_cohort["start_date"].isna().sum(),
        primary_hf_cohort["completion_date"].isna().sum()
    ]
})

date_availability["percentage_available"] = (
    date_availability["available_trials"]
    / len(primary_hf_cohort)
    * 100
).round(2)

date_availability

In [ ]:
# Assess start-date and completion-date availability by trial status

date_availability_by_status = (
    primary_hf_cohort
    .groupby("overall_status")
    .agg(
        total_trials=("nct_id", "nunique"),
        start_date_available=("start_date", "count"),
        completion_date_available=("completion_date", "count")
    )
    .reset_index()
)

date_availability_by_status["start_date_available_pct"] = (
    date_availability_by_status["start_date_available"]
    / date_availability_by_status["total_trials"]
    * 100
).round(2)

date_availability_by_status["completion_date_available_pct"] = (
    date_availability_by_status["completion_date_available"]
    / date_availability_by_status["total_trials"]
    * 100
).round(2)

date_availability_by_status = (
    date_availability_by_status
    .sort_values("total_trials", ascending=False)
    .reset_index(drop=True)
)

date_availability_by_status

In [ ]:
# Create a historical-status subset for timeline assessment

historical_statuses = [
    "COMPLETED",
    "TERMINATED",
    "WITHDRAWN"
]

historical_hf_trials = (
    primary_hf_cohort[
        primary_hf_cohort["overall_status"].isin(historical_statuses)
    ]
    .copy()
)

# Convert dates to datetime
historical_hf_trials["start_date_dt"] = pd.to_datetime(
    historical_hf_trials["start_date"],
    errors="coerce"
)

historical_hf_trials["completion_date_dt"] = pd.to_datetime(
    historical_hf_trials["completion_date"],
    errors="coerce"
)

# Calculate duration in days
historical_hf_trials["trial_duration_days"] = (
    historical_hf_trials["completion_date_dt"]
    - historical_hf_trials["start_date_dt"]
).dt.days

print("Historical-status trials:", len(historical_hf_trials))

print(
    "Trials with calculable duration:",
    historical_hf_trials["trial_duration_days"].notna().sum()
)

print(
    "Trials with negative duration:",
    (historical_hf_trials["trial_duration_days"] < 0).sum()
)

historical_hf_trials[
    "trial_duration_days"
].describe()

In [ ]:
# Inspect trials with impossible negative calculated durations

negative_duration_trials = (
    historical_hf_trials[
        historical_hf_trials["trial_duration_days"] < 0
    ][
        [
            "nct_id",
            "brief_title",
            "overall_status",
            "start_date",
            "completion_date",
            "trial_duration_days"
        ]
    ]
    .sort_values("trial_duration_days")
)

negative_duration_trials

In [ ]:
# Mark impossible negative trial durations as missing

historical_hf_trials.loc[
    historical_hf_trials["trial_duration_days"] < 0,
    "trial_duration_days"
] = np.nan

print(
    "Valid calculable durations:",
    historical_hf_trials["trial_duration_days"].notna().sum()
)

print(
    "Remaining negative durations:",
    (historical_hf_trials["trial_duration_days"] < 0).sum()
)

In [ ]:
# Compare trial-duration distributions by historical trial status

duration_by_status = (
    historical_hf_trials
    .groupby("overall_status")["trial_duration_days"]
    .agg(
        valid_trials="count",
        mean_days="mean",
        median_days="median",
        min_days="min",
        max_days="max"
    )
    .round(1)
    .reset_index()
)

# Add duration in years for easier interpretation
duration_by_status["mean_years"] = (
    duration_by_status["mean_days"] / 365.25
).round(2)

duration_by_status["median_years"] = (
    duration_by_status["median_days"] / 365.25
).round(2)

duration_by_status

In [ ]:
# Examine duration distribution among completed HF trials only

completed_hf_trials = (
    historical_hf_trials[
        (historical_hf_trials["overall_status"] == "COMPLETED")
        & historical_hf_trials["trial_duration_days"].notna()
    ]
    .copy()
)

completed_duration_quantiles = (
    completed_hf_trials["trial_duration_days"]
    .quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .reset_index()
)

completed_duration_quantiles.columns = [
    "quantile",
    "duration_days"
]

completed_duration_quantiles["duration_years"] = (
    completed_duration_quantiles["duration_days"] / 365.25
).round(2)

print(
    "Completed trials with valid duration:",
    len(completed_hf_trials)
)

completed_duration_quantiles

In [ ]:
# Examine completed-trial duration by phase

completed_duration_by_phase = (
    completed_hf_trials
    .assign(
        phase_display=lambda df: (
            df["phase"]
            .replace("NA", "NA / Not Applicable or Not Reported")
            .fillna("NA / Not Applicable or Not Reported")
        )
    )
    .groupby("phase_display")["trial_duration_days"]
    .agg(
        valid_trials="count",
        median_days="median",
        mean_days="mean",
        q75_days=lambda x: x.quantile(0.75),
        q90_days=lambda x: x.quantile(0.90)
    )
    .round(1)
    .reset_index()
)

completed_duration_by_phase["median_years"] = (
    completed_duration_by_phase["median_days"] / 365.25
).round(2)

completed_duration_by_phase["q75_years"] = (
    completed_duration_by_phase["q75_days"] / 365.25
).round(2)

completed_duration_by_phase = (
    completed_duration_by_phase
    .sort_values("median_days")
    .reset_index(drop=True)
)

completed_duration_by_phase

In [ ]:
# Compare completed-trial duration by intervention type

completed_intervention_duration = (
    primary_hf_interventions[
        primary_hf_interventions["nct_id"].isin(
            completed_hf_trials["nct_id"]
        )
    ][["nct_id", "intervention_type"]]
    .drop_duplicates()
    .merge(
        completed_hf_trials[
            ["nct_id", "trial_duration_days"]
        ],
        on="nct_id",
        how="inner"
    )
)

duration_by_intervention_type = (
    completed_intervention_duration
    .groupby("intervention_type")["trial_duration_days"]
    .agg(
        valid_trials="count",
        median_days="median",
        mean_days="mean",
        q75_days=lambda x: x.quantile(0.75),
        q90_days=lambda x: x.quantile(0.90)
    )
    .round(1)
    .reset_index()
)

duration_by_intervention_type["median_years"] = (
    duration_by_intervention_type["median_days"] / 365.25
).round(2)

duration_by_intervention_type["q75_years"] = (
    duration_by_intervention_type["q75_days"] / 365.25
).round(2)

duration_by_intervention_type = (
    duration_by_intervention_type
    .sort_values("valid_trials", ascending=False)
    .reset_index(drop=True)
)

duration_by_intervention_type

In [ ]:
# Check whether AACT provides actual/estimated indicators for trial dates

date_type_query = text("""
SELECT
    nct_id,
    overall_status,
    start_date,
    start_date_type,
    completion_date,
    completion_date_type
FROM ctgov.studies
WHERE nct_id = ANY(:nct_ids);
""")

date_type_check = pd.read_sql_query(
    date_type_query,
    engine,
    params={
        "nct_ids": historical_hf_trials["nct_id"].tolist()
    }
)

print("Rows retrieved:", len(date_type_check))

print("\nStart-date type:")
print(
    date_type_check["start_date_type"]
    .fillna("MISSING")
    .value_counts(dropna=False)
)

print("\nCompletion-date type:")
print(
    date_type_check["completion_date_type"]
    .fillna("MISSING")
    .value_counts(dropna=False)
)

In [ ]:
# Examine date-type quality specifically among completed HF trials

completed_date_quality = (
    completed_hf_trials[
        ["nct_id", "trial_duration_days"]
    ]
    .merge(
        date_type_check[
            [
                "nct_id",
                "start_date_type",
                "completion_date_type"
            ]
        ],
        on="nct_id",
        how="left"
    )
)

print("Completed trials with valid duration:", len(completed_date_quality))

print("\nCompletion-date type among completed trials:")
print(
    completed_date_quality["completion_date_type"]
    .fillna("MISSING")
    .value_counts(dropna=False)
)

print("\nStart-date type among completed trials:")
print(
    completed_date_quality["start_date_type"]
    .fillna("MISSING")
    .value_counts(dropna=False)
)

print("\nCross-tabulation of start and completion date types:")
print(
    pd.crosstab(
        completed_date_quality["start_date_type"].fillna("MISSING"),
        completed_date_quality["completion_date_type"].fillna("MISSING"),
        margins=True
    )
)

In [ ]:
# Create the primary cohort for completed-trial duration modeling

duration_model_cohort = (
    completed_date_quality[
        completed_date_quality["completion_date_type"] == "ACTUAL"
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Trials in duration-modeling cohort:",
    len(duration_model_cohort)
)

# Calculate duration distribution for the final duration cohort
duration_model_quantiles = (
    duration_model_cohort["trial_duration_days"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .reset_index()
)

duration_model_quantiles.columns = [
    "quantile",
    "duration_days"
]

duration_model_quantiles["duration_years"] = (
    duration_model_quantiles["duration_days"] / 365.25
).round(2)

duration_model_quantiles

In [ ]:
# Define the primary unusually-long-duration outcome

LONG_DURATION_THRESHOLD = 1461.0

duration_model_cohort["long_duration"] = (
    duration_model_cohort["trial_duration_days"]
    > LONG_DURATION_THRESHOLD
).astype(int)

# Examine target distribution
long_duration_distribution = (
    duration_model_cohort["long_duration"]
    .value_counts()
    .rename_axis("long_duration")
    .reset_index(name="trial_count")
)

long_duration_distribution["label"] = (
    long_duration_distribution["long_duration"]
    .map({
        0: "Normal/shorter duration",
        1: "Unusually long duration"
    })
)

long_duration_distribution["percentage"] = (
    long_duration_distribution["trial_count"]
    / len(duration_model_cohort)
    * 100
).round(2)

print(
    "Long-duration threshold:",
    LONG_DURATION_THRESHOLD,
    "days"
)

print(
    "Threshold in years:",
    round(LONG_DURATION_THRESHOLD / 365.25, 2)
)

long_duration_distribution[
    ["long_duration", "label", "trial_count", "percentage"]
]

In [ ]:
# Create the historical cohort for trial-discontinuation modeling

discontinuation_model_cohort = (
    historical_hf_trials[
        historical_hf_trials["overall_status"].isin(
            ["COMPLETED", "TERMINATED", "WITHDRAWN"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Binary outcome:
# 0 = Completed
# 1 = Terminated or Withdrawn

discontinuation_model_cohort["discontinued"] = (
    discontinuation_model_cohort["overall_status"]
    .isin(["TERMINATED", "WITHDRAWN"])
    .astype(int)
)

discontinuation_distribution = (
    discontinuation_model_cohort["discontinued"]
    .value_counts()
    .rename_axis("discontinued")
    .reset_index(name="trial_count")
)

discontinuation_distribution["label"] = (
    discontinuation_distribution["discontinued"]
    .map({
        0: "Completed",
        1: "Terminated/Withdrawn"
    })
)

discontinuation_distribution["percentage"] = (
    discontinuation_distribution["trial_count"]
    / len(discontinuation_model_cohort)
    * 100
).round(2)

print(
    "Trials in discontinuation-modeling cohort:",
    len(discontinuation_model_cohort)
)

discontinuation_distribution[
    ["discontinued", "label", "trial_count", "percentage"]
]

In [ ]:
# Inspect available columns in the AACT designs table

design_columns_query = text("""
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'ctgov'
  AND table_name = 'designs'
ORDER BY ordinal_position;
""")

design_columns = pd.read_sql_query(
    design_columns_query,
    engine
)

design_columns

In [ ]:
# Retrieve planned trial-design characteristics
# for the discontinuation-modeling cohort

design_query = text("""
SELECT
    nct_id,
    allocation,
    intervention_model,
    primary_purpose,
    masking,
    subject_masked,
    caregiver_masked,
    investigator_masked,
    outcomes_assessor_masked
FROM ctgov.designs
WHERE nct_id = ANY(:nct_ids);
""")

hf_design_features = pd.read_sql_query(
    design_query,
    engine,
    params={
        "nct_ids": discontinuation_model_cohort["nct_id"].tolist()
    }
)

print("Design records retrieved:", len(hf_design_features))
print(
    "Unique trials with design records:",
    hf_design_features["nct_id"].nunique()
)

hf_design_features.head()

In [ ]:
# Examine missingness and category distributions
# in the retrieved trial-design variables

design_variables = [
    "allocation",
    "intervention_model",
    "primary_purpose",
    "masking",
    "subject_masked",
    "caregiver_masked",
    "investigator_masked",
    "outcomes_assessor_masked"
]

design_missingness = pd.DataFrame({
    "variable": design_variables,
    "missing_count": [
        hf_design_features[col].isna().sum()
        for col in design_variables
    ],
    "missing_percentage": [
        round(
            hf_design_features[col].isna().mean() * 100,
            2
        )
        for col in design_variables
    ]
})

design_missingness

In [ ]:
# Inspect category distributions for the main trial-design variables

for col in [
    "allocation",
    "intervention_model",
    "primary_purpose",
    "masking"
]:
    print(f"\n{'=' * 60}")
    print(col.upper())
    print("=" * 60)

    counts = (
        hf_design_features[col]
        .fillna("MISSING")
        .value_counts(dropna=False)
    )

    print(counts)

In [ ]:
# Inspect available columns in the AACT sponsors table

sponsor_columns_query = text("""
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'ctgov'
  AND table_name = 'sponsors'
ORDER BY ordinal_position;
""")

sponsor_columns = pd.read_sql_query(
    sponsor_columns_query,
    engine
)

sponsor_columns

In [ ]:
# Retrieve lead-sponsor information for the discontinuation cohort

lead_sponsor_query = text("""
SELECT
    nct_id,
    agency_class,
    name AS lead_sponsor_name
FROM ctgov.sponsors
WHERE nct_id = ANY(:nct_ids)
  AND lead_or_collaborator = 'lead';
""")

hf_lead_sponsors = pd.read_sql_query(
    lead_sponsor_query,
    engine,
    params={
        "nct_ids": discontinuation_model_cohort["nct_id"].tolist()
    }
)

print("Lead-sponsor records retrieved:", len(hf_lead_sponsors))
print(
    "Unique trials with lead-sponsor records:",
    hf_lead_sponsors["nct_id"].nunique()
)

print("\nLead sponsor agency classes:")
print(
    hf_lead_sponsors["agency_class"]
    .fillna("MISSING")
    .value_counts(dropna=False)
)

hf_lead_sponsors.head()

In [ ]:
# Inspect available columns in the AACT facilities table

facility_columns_query = text("""
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'ctgov'
  AND table_name = 'facilities'
ORDER BY ordinal_position;
""")

facility_columns = pd.read_sql_query(
    facility_columns_query,
    engine
)

facility_columns

In [ ]:
# Create trial-level geographic/site features

facility_query = text("""
SELECT
    nct_id,
    name,
    country
FROM ctgov.facilities
WHERE nct_id = ANY(:nct_ids);
""")

hf_facilities = pd.read_sql_query(
    facility_query,
    engine,
    params={
        "nct_ids": discontinuation_model_cohort["nct_id"].tolist()
    }
)

print("Facility records retrieved:", len(hf_facilities))
print(
    "Trials with at least one facility record:",
    hf_facilities["nct_id"].nunique()
)

# Aggregate facility information to one row per trial
hf_geography_features = (
    hf_facilities
    .groupby("nct_id")
    .agg(
        number_of_sites=("name", "nunique"),
        number_of_countries=("country", "nunique")
    )
    .reset_index()
)

# Multinational indicator
hf_geography_features["multinational_trial"] = (
    hf_geography_features["number_of_countries"] > 1
).astype(int)

print(
    "\nTrials with derived geography features:",
    hf_geography_features["nct_id"].nunique()
)

hf_geography_features[
    [
        "number_of_sites",
        "number_of_countries"
    ]
].describe()

In [ ]:
# Investigate trials that have facility records
# but were calculated as having zero named sites

zero_site_trials = hf_geography_features[
    hf_geography_features["number_of_sites"] == 0
]["nct_id"].tolist()

print("Trials calculated as zero sites:", len(zero_site_trials))

zero_site_facilities = (
    hf_facilities[
        hf_facilities["nct_id"].isin(zero_site_trials)
    ]
    .sort_values("nct_id")
)

zero_site_facilities.head(20)

In [ ]:
# Rebuild trial-level geographic/site features using location information

facility_query = text("""
SELECT
    nct_id,
    name,
    city,
    state,
    country
FROM ctgov.facilities
WHERE nct_id = ANY(:nct_ids);
""")

hf_facilities = pd.read_sql_query(
    facility_query,
    engine,
    params={
        "nct_ids": discontinuation_model_cohort["nct_id"].tolist()
    }
)

# Create a location-based site identifier.
# Missing components are replaced only for constructing the identifier.
hf_facilities["site_identifier"] = (
    hf_facilities["name"].fillna("UNKNOWN_NAME").astype(str)
    + " | "
    + hf_facilities["city"].fillna("UNKNOWN_CITY").astype(str)
    + " | "
    + hf_facilities["state"].fillna("UNKNOWN_STATE").astype(str)
    + " | "
    + hf_facilities["country"].fillna("UNKNOWN_COUNTRY").astype(str)
)

# Collapse to one row per trial
hf_geography_features = (
    hf_facilities
    .groupby("nct_id")
    .agg(
        number_of_sites=("site_identifier", "nunique"),
        number_of_countries=("country", "nunique")
    )
    .reset_index()
)

hf_geography_features["multinational_trial"] = (
    hf_geography_features["number_of_countries"] > 1
).astype(int)

print("Facility records:", len(hf_facilities))

print(
    "Trials with facility information:",
    hf_geography_features["nct_id"].nunique()
)

print(
    "Trials calculated as zero sites:",
    (hf_geography_features["number_of_sites"] == 0).sum()
)

hf_geography_features[
    ["number_of_sites", "number_of_countries"]
].describe()

In [ ]:
# Inspect trials with the largest reported number of sites

largest_site_trials = (
    hf_geography_features
    .sort_values("number_of_sites", ascending=False)
    .head(15)
    .merge(
        discontinuation_model_cohort[
            [
                "nct_id",
                "brief_title",
                "overall_status",
                "phase",
                "enrollment"
            ]
        ],
        on="nct_id",
        how="left"
    )
)

largest_site_trials[
    [
        "nct_id",
        "brief_title",
        "overall_status",
        "phase",
        "enrollment",
        "number_of_sites",
        "number_of_countries",
        "multinational_trial"
    ]
]

In [ ]:
# Inspect available columns in the AACT eligibilities table

eligibility_columns_query = text("""
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'ctgov'
  AND table_name = 'eligibilities'
ORDER BY ordinal_position;
""")

eligibility_columns = pd.read_sql_query(
    eligibility_columns_query,
    engine
)

eligibility_columns

In [ ]:
# Retrieve eligibility information for the discontinuation cohort

eligibility_query = text("""
SELECT
    nct_id,
    gender,
    minimum_age,
    maximum_age,
    healthy_volunteers,
    adult,
    child,
    older_adult,
    criteria
FROM ctgov.eligibilities
WHERE nct_id = ANY(:nct_ids);
""")

hf_eligibility = pd.read_sql_query(
    eligibility_query,
    engine,
    params={
        "nct_ids": discontinuation_model_cohort["nct_id"].tolist()
    }
)

print("Eligibility records retrieved:", len(hf_eligibility))
print(
    "Unique trials with eligibility records:",
    hf_eligibility["nct_id"].nunique()
)

print("\nMissingness:")
print(
    hf_eligibility[
        [
            "gender",
            "minimum_age",
            "maximum_age",
            "healthy_volunteers",
            "adult",
            "child",
            "older_adult",
            "criteria"
        ]
    ]
    .isna()
    .sum()
)

print("\nMost common minimum-age values:")
print(
    hf_eligibility["minimum_age"]
    .fillna("MISSING")
    .value_counts()
    .head(15)
)

print("\nMost common maximum-age values:")
print(
    hf_eligibility["maximum_age"]
    .fillna("MISSING")
    .value_counts()
    .head(15)
)

In [ ]:
# Inspect the units used in minimum_age and maximum_age

def extract_age_unit(value):
    if pd.isna(value):
        return "MISSING"

    value = str(value).strip()

    # Return everything after the numeric portion
    match = re.match(r"^\s*[\d.]+\s*(.*)$", value)

    if match:
        unit = match.group(1).strip()
        return unit if unit else "NO_UNIT"

    return value


print("Minimum-age units:")
print(
    hf_eligibility["minimum_age"]
    .apply(extract_age_unit)
    .value_counts(dropna=False)
)

print("\nMaximum-age units:")
print(
    hf_eligibility["maximum_age"]
    .apply(extract_age_unit)
    .value_counts(dropna=False)
)

In [ ]:
# Convert ClinicalTrials.gov age strings to years

def age_to_years(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    match = re.match(
        r"^\s*([\d.]+)\s*(Hours?|Days?|Weeks?|Months?|Years?)\s*$",
        value,
        flags=re.IGNORECASE
    )

    if not match:
        return np.nan

    number = float(match.group(1))
    unit = match.group(2).lower()

    if unit.startswith("year"):
        return number
    elif unit.startswith("month"):
        return number / 12
    elif unit.startswith("week"):
        return number / 52.1429
    elif unit.startswith("day"):
        return number / 365.25
    elif unit.startswith("hour"):
        return number / (365.25 * 24)

    return np.nan


hf_eligibility["minimum_age_years"] = (
    hf_eligibility["minimum_age"].apply(age_to_years)
)

hf_eligibility["maximum_age_years"] = (
    hf_eligibility["maximum_age"].apply(age_to_years)
)

# Indicator showing whether an explicit upper-age limit was reported
hf_eligibility["maximum_age_reported"] = (
    hf_eligibility["maximum_age"].notna()
).astype(int)


print("Minimum age after conversion:")
print(hf_eligibility["minimum_age_years"].describe())

print("\nMaximum age after conversion:")
print(hf_eligibility["maximum_age_years"].describe())

print("\nMaximum-age limit reported:")
print(
    hf_eligibility["maximum_age_reported"]
    .value_counts()
    .rename(index={0: "No explicit maximum", 1: "Maximum reported"})
)

print("\nUnconverted non-missing minimum ages:",
      (
          hf_eligibility["minimum_age"].notna()
          & hf_eligibility["minimum_age_years"].isna()
      ).sum())

print("Unconverted non-missing maximum ages:",
      (
          hf_eligibility["maximum_age"].notna()
          & hf_eligibility["maximum_age_years"].isna()
      ).sum())

In [ ]:
# Inspect distributions of structured eligibility characteristics

print("Gender:")
print(
    hf_eligibility["gender"]
    .fillna("MISSING")
    .value_counts(dropna=False)
)

print("\nHealthy volunteers:")
print(
    hf_eligibility["healthy_volunteers"]
    .fillna("MISSING")
    .value_counts(dropna=False)
)

print("\nAdult eligibility:")
print(
    hf_eligibility["adult"]
    .value_counts(dropna=False)
)

print("\nChild eligibility:")
print(
    hf_eligibility["child"]
    .value_counts(dropna=False)
)

print("\nOlder-adult eligibility:")
print(
    hf_eligibility["older_adult"]
    .value_counts(dropna=False)
)

In [ ]:
# Inspect available columns in the AACT design_outcomes table

outcome_columns_query = text("""
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'ctgov'
  AND table_name = 'design_outcomes'
ORDER BY ordinal_position;
""")

outcome_columns = pd.read_sql_query(
    outcome_columns_query,
    engine
)

outcome_columns

In [ ]:
# Retrieve planned outcome/end-point information
# for the discontinuation modeling cohort

design_outcomes_query = text("""
SELECT
    nct_id,
    outcome_type,
    measure,
    time_frame,
    description
FROM ctgov.design_outcomes
WHERE nct_id = ANY(:nct_ids);
""")

hf_design_outcomes = pd.read_sql_query(
    design_outcomes_query,
    engine,
    params={
        "nct_ids": discontinuation_model_cohort["nct_id"].tolist()
    }
)

print("Outcome records retrieved:", len(hf_design_outcomes))

print(
    "Trials with at least one outcome record:",
    hf_design_outcomes["nct_id"].nunique()
)

print(
    "Trials with no outcome records:",
    len(discontinuation_model_cohort)
    - hf_design_outcomes["nct_id"].nunique()
)

print("\nOutcome-type distribution:")
print(
    hf_design_outcomes["outcome_type"]
    .fillna("MISSING")
    .value_counts(dropna=False)
)

print("\nMissingness:")
print(
    hf_design_outcomes[
        [
            "outcome_type",
            "measure",
            "time_frame",
            "description"
        ]
    ]
    .isna()
    .sum()
)

In [ ]:
# Create trial-level endpoint complexity features

outcome_counts = (
    hf_design_outcomes
    .groupby(["nct_id", "outcome_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure expected columns exist
for col in ["primary", "secondary", "other"]:
    if col not in outcome_counts.columns:
        outcome_counts[col] = 0

# Rename for clarity
outcome_counts = outcome_counts.rename(
    columns={
        "primary": "number_of_primary_outcomes",
        "secondary": "number_of_secondary_outcomes",
        "other": "number_of_other_outcomes"
    }
)

# Total number of reported planned outcomes
outcome_counts["total_number_of_outcomes"] = (
    outcome_counts["number_of_primary_outcomes"]
    + outcome_counts["number_of_secondary_outcomes"]
    + outcome_counts["number_of_other_outcomes"]
)

print("Trials with outcome-count features:", outcome_counts["nct_id"].nunique())

print("\nEndpoint-count distributions:")
print(
    outcome_counts[
        [
            "number_of_primary_outcomes",
            "number_of_secondary_outcomes",
            "number_of_other_outcomes",
            "total_number_of_outcomes"
        ]
    ].describe()
)

print("\nTrials with the largest numbers of reported outcomes:")
display(
    outcome_counts
    .sort_values("total_number_of_outcomes", ascending=False)
    .head(10)
)

In [ ]:
# Inspect enrollment-related and date/submission fields
# available in the AACT studies table

studies_columns_query = text("""
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'ctgov'
  AND table_name = 'studies'
  AND (
       column_name ILIKE '%enrollment%'
       OR column_name ILIKE '%submitted%'
       OR column_name ILIKE '%start_date%'
       OR column_name ILIKE '%study_first%'
       OR column_name ILIKE '%phase%'
  )
ORDER BY ordinal_position;
""")

studies_relevant_columns = pd.read_sql_query(
    studies_columns_query,
    engine
)

studies_relevant_columns

In [ ]:
# Retrieve enrollment and temporal fields for the discontinuation cohort

enrollment_query = text("""
SELECT
    nct_id,
    enrollment,
    enrollment_type,
    study_first_submitted_date,
    study_first_posted_date,
    start_date,
    start_date_type
FROM ctgov.studies
WHERE nct_id = ANY(:nct_ids);
""")

hf_enrollment_temporal = pd.read_sql_query(
    enrollment_query,
    engine,
    params={
        "nct_ids": discontinuation_model_cohort["nct_id"].tolist()
    }
)

# Add outcome status for the leakage audit
hf_enrollment_temporal = hf_enrollment_temporal.merge(
    discontinuation_model_cohort[
        ["nct_id", "overall_status", "discontinued"]
    ],
    on="nct_id",
    how="left",
    validate="one_to_one"
)

print("Trials retrieved:", len(hf_enrollment_temporal))
print(
    "Unique trials:",
    hf_enrollment_temporal["nct_id"].nunique()
)

print("\nEnrollment type:")
print(
    hf_enrollment_temporal["enrollment_type"]
    .fillna("MISSING")
    .value_counts()
)

print("\nEnrollment type by trial status:")
print(
    pd.crosstab(
        hf_enrollment_temporal["overall_status"],
        hf_enrollment_temporal["enrollment_type"].fillna("MISSING"),
        margins=True
    )
)

print("\nMissingness:")
print(
    hf_enrollment_temporal[
        [
            "enrollment",
            "enrollment_type",
            "study_first_submitted_date",
            "study_first_posted_date",
            "start_date",
            "start_date_type"
        ]
    ].isna().sum()
)

In [ ]:
# Create submission-year feature and inspect the historical distribution

hf_enrollment_temporal["submission_year"] = pd.to_datetime(
    hf_enrollment_temporal["study_first_submitted_date"]
).dt.year

year_summary = (
    hf_enrollment_temporal
    .groupby("submission_year")
    .agg(
        number_of_trials=("nct_id", "count"),
        discontinued_trials=("discontinued", "sum"),
        discontinuation_rate=("discontinued", "mean")
    )
    .reset_index()
)

year_summary["discontinuation_rate_percent"] = (
    year_summary["discontinuation_rate"] * 100
)

print("Submission-year range:")
print(
    hf_enrollment_temporal["submission_year"].min(),
    "to",
    hf_enrollment_temporal["submission_year"].max()
)

print("\nTrials by submission year:")
display(
    year_summary[
        [
            "submission_year",
            "number_of_trials",
            "discontinued_trials",
            "discontinuation_rate_percent"
        ]
    ]
)

print("\nSubmission-year quantiles:")
print(
    hf_enrollment_temporal["submission_year"]
    .quantile([0.25, 0.50, 0.75, 0.80, 0.90])
)

In [ ]:
# Evaluate a candidate temporal split:
# Train = submitted through 2020
# Test  = submitted from 2021 onward

TEMPORAL_CUTOFF_YEAR = 2020

hf_enrollment_temporal["temporal_set"] = np.where(
    hf_enrollment_temporal["submission_year"] <= TEMPORAL_CUTOFF_YEAR,
    "Train (≤2020)",
    "Test (2021+)"
)

temporal_split_summary = (
    hf_enrollment_temporal
    .groupby("temporal_set")
    .agg(
        number_of_trials=("nct_id", "count"),
        completed_trials=("discontinued", lambda x: (x == 0).sum()),
        discontinued_trials=("discontinued", lambda x: (x == 1).sum()),
        discontinuation_rate=("discontinued", "mean"),
        earliest_year=("submission_year", "min"),
        latest_year=("submission_year", "max")
    )
    .reset_index()
)

temporal_split_summary["discontinuation_rate_percent"] = (
    temporal_split_summary["discontinuation_rate"] * 100
)

temporal_split_summary[
    [
        "temporal_set",
        "number_of_trials",
        "completed_trials",
        "discontinued_trials",
        "discontinuation_rate_percent",
        "earliest_year",
        "latest_year"
    ]
]

In [ ]:
# Examine outcome maturity within the proposed 2021+ temporal test set

test_year_status = (
    hf_enrollment_temporal[
        hf_enrollment_temporal["submission_year"] >= 2021
    ]
    .groupby(["submission_year", "overall_status"])
    .size()
    .unstack(fill_value=0)
)

# Ensure expected status columns exist
for col in ["COMPLETED", "TERMINATED", "WITHDRAWN"]:
    if col not in test_year_status.columns:
        test_year_status[col] = 0

test_year_status["total"] = (
    test_year_status["COMPLETED"]
    + test_year_status["TERMINATED"]
    + test_year_status["WITHDRAWN"]
)

test_year_status["discontinued"] = (
    test_year_status["TERMINATED"]
    + test_year_status["WITHDRAWN"]
)

test_year_status["discontinuation_rate_percent"] = (
    test_year_status["discontinued"]
    / test_year_status["total"]
    * 100
)

test_year_status[
    [
        "COMPLETED",
        "TERMINATED",
        "WITHDRAWN",
        "total",
        "discontinued",
        "discontinuation_rate_percent"
    ]
]

In [ ]:
# Define the primary temporal evaluation framework
#
# Training: trials first submitted through 2020
# Test:     trials first submitted from 2021 through 2024
# Holdout from primary evaluation: 2025-2026
# because outcomes are likely insufficiently mature

def assign_temporal_group(year):
    if year <= 2020:
        return "Train (≤2020)"
    elif 2021 <= year <= 2024:
        return "Test (2021–2024)"
    else:
        return "Outcome-immature (2025–2026)"


hf_enrollment_temporal["modeling_period"] = (
    hf_enrollment_temporal["submission_year"]
    .apply(assign_temporal_group)
)

final_temporal_summary = (
    hf_enrollment_temporal
    .groupby("modeling_period")
    .agg(
        number_of_trials=("nct_id", "count"),
        completed_trials=("discontinued", lambda x: (x == 0).sum()),
        discontinued_trials=("discontinued", lambda x: (x == 1).sum()),
        discontinuation_rate=("discontinued", "mean"),
        earliest_year=("submission_year", "min"),
        latest_year=("submission_year", "max")
    )
    .reset_index()
)

final_temporal_summary["discontinuation_rate_percent"] = (
    final_temporal_summary["discontinuation_rate"] * 100
)

final_temporal_summary[
    [
        "modeling_period",
        "number_of_trials",
        "completed_trials",
        "discontinued_trials",
        "discontinuation_rate_percent",
        "earliest_year",
        "latest_year"
    ]
]

In [ ]:
# STEP 1A — Verify current HeartTrialTwin modeling cohort

print("Dataset shape:", hf_enrollment_temporal.shape)
print("Unique trials:", hf_enrollment_temporal["nct_id"].nunique())

print("\nOutcome counts:")
print(
    hf_enrollment_temporal["discontinued"]
    .value_counts()
    .sort_index()
)

print("\nTemporal groups:")
print(
    hf_enrollment_temporal["modeling_period"]
    .value_counts()
)

print("\nSubmission-year range:")
print(
    hf_enrollment_temporal["submission_year"].min(),
    "to",
    hf_enrollment_temporal["submission_year"].max()
)

print("\nAvailable columns:")
print(hf_enrollment_temporal.columns.tolist())

In [ ]:
# STEP 1B — Audit outcome definition and temporal distribution

import pandas as pd

# 1. Exact current status counts
print("=== OVERALL STATUS COUNTS ===")
print(
    hf_enrollment_temporal["overall_status"]
    .value_counts(dropna=False)
)

# 2. Cross-check status against binary outcome
print("\n=== STATUS vs DISCONTINUED LABEL ===")
print(
    pd.crosstab(
        hf_enrollment_temporal["overall_status"],
        hf_enrollment_temporal["discontinued"],
        margins=True
    )
)

# 3. Outcome distribution by submission year
yearly_outcomes = (
    hf_enrollment_temporal
    .groupby("submission_year")
    .agg(
        total_trials=("nct_id", "count"),
        completed=("discontinued", lambda x: (x == 0).sum()),
        discontinued=("discontinued", lambda x: (x == 1).sum()),
        discontinuation_rate=("discontinued", "mean")
    )
    .reset_index()
)

yearly_outcomes["discontinuation_rate_percent"] = (
    yearly_outcomes["discontinuation_rate"] * 100
).round(1)

display(
    yearly_outcomes[
        [
            "submission_year",
            "total_trials",
            "completed",
            "discontinued",
            "discontinuation_rate_percent"
        ]
    ]
)

In [ ]:
# STEP 1C — Identify earlier cohort dataframes

import pandas as pd

print("=== DATAFRAMES CURRENTLY IN MEMORY ===\n")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(
            f"{name:35s} "
            f"rows={len(obj):6d}  "
            f"cols={obj.shape[1]:3d}"
        )

In [ ]:
# STEP 1D — Reconstruct HeartTrialTwin HF cohort flow

cohort_flow = pd.DataFrame({
    "stage": [
        "HF candidate rows",
        "Unique HF trials",
        "Primary HF cohort",
        "Excluded from primary HF cohort",
        "Terminal-outcome historical cohort",
        "Discontinuation modeling cohort"
    ],
    "dataframe": [
        "hf_candidates",
        "hf_one_row",
        "primary_hf_cohort",
        "excluded_from_primary_hf",
        "historical_hf_trials",
        "discontinuation_model_cohort"
    ],
    "rows": [
        len(hf_candidates),
        len(hf_one_row),
        len(primary_hf_cohort),
        len(excluded_from_primary_hf),
        len(historical_hf_trials),
        len(discontinuation_model_cohort)
    ],
    "unique_nct_ids": [
        hf_candidates["nct_id"].nunique(),
        hf_one_row["nct_id"].nunique(),
        primary_hf_cohort["nct_id"].nunique(),
        excluded_from_primary_hf["nct_id"].nunique(),
        historical_hf_trials["nct_id"].nunique(),
        discontinuation_model_cohort["nct_id"].nunique()
    ]
})

display(cohort_flow)

print("\n=== STATUS DISTRIBUTION: PRIMARY HF COHORT ===")
print(primary_hf_cohort["overall_status"].value_counts(dropna=False))

print("\n=== STATUS DISTRIBUTION: 2,901 MODELING COHORT ===")
print(discontinuation_model_cohort["overall_status"].value_counts(dropna=False))

In [ ]:
# STEP 1E — Understand the 150 trials excluded from the primary HF cohort

print("Excluded trials:", len(excluded_from_primary_hf))
print("Unique excluded NCT IDs:",
      excluded_from_primary_hf["nct_id"].nunique())

print("\n=== AVAILABLE COLUMNS ===")
print(excluded_from_primary_hf.columns.tolist())

print("\n=== FIRST 10 EXCLUDED TRIALS ===")

display(
    excluded_from_primary_hf.head(10)
)

In [ ]:
# STEP 1F — Quantify primary heart-failure inclusion evidence

evidence_cols = [
    "hf_in_title",
    "hf_in_condition",
    "hf_in_mesh",
    "hf_in_summary",
    "hf_in_condition_expanded",
    "hf_in_text_expanded",
    "hf_evidence_score",
    "primary_hf_inclusion"
]

print("=== ALL 4,738 CANDIDATE HF TRIALS ===")

for col in evidence_cols:
    if col in hf_cohort.columns:
        print(f"\n{col}:")
        print(hf_cohort[col].value_counts(dropna=False).sort_index())


print("\n\n=== EXCLUDED 150 TRIALS: EVIDENCE PATTERNS ===")

excluded_patterns = (
    excluded_from_primary_hf[
        [
            "hf_in_title",
            "hf_in_condition",
            "hf_in_mesh",
            "hf_in_summary",
            "hf_in_condition_expanded",
            "hf_in_text_expanded",
            "hf_evidence_score"
        ]
    ]
    .value_counts(dropna=False)
    .reset_index(name="number_of_trials")
)

display(excluded_patterns)

In [ ]:
# STEP 2A — Inventory candidate registration-time feature tables
# No merging, feature selection, or modeling yet.

feature_tables = {
    "design": hf_design_features,
    "sponsor": hf_lead_sponsors,
    "geography": hf_geography_features,
    "eligibility": hf_eligibility,
}

for table_name, df in feature_tables.items():

    print("\n" + "=" * 80)
    print(f"{table_name.upper()} FEATURES")
    print("=" * 80)

    print(f"Shape: {df.shape}")
    print(f"Unique trials: {df['nct_id'].nunique()}")

    print("\nColumns:")
    for col in df.columns:
        print(f"  - {col}")

    print("\nMissingness:")
    missing = (
        df.isna()
        .mean()
        .mul(100)
        .round(1)
        .sort_values(ascending=False)
    )

    print(missing.to_string())

In [ ]:
# STEP 2B — Inspect candidate registration-time feature values
# Descriptive audit only — no modeling.

print("=== TRIAL DESIGN ===")

for col in [
    "allocation",
    "intervention_model",
    "primary_purpose",
    "masking"
]:
    print(f"\n{col}:")
    print(
        hf_design_features[col]
        .value_counts(dropna=False)
        .head(20)
    )


print("\n\n=== SPONSOR CLASS ===")
print(
    hf_lead_sponsors["agency_class"]
    .value_counts(dropna=False)
)


print("\n\n=== ELIGIBILITY DEMOGRAPHICS ===")

for col in [
    "gender",
    "healthy_volunteers",
    "adult",
    "child",
    "older_adult"
]:
    print(f"\n{col}:")
    print(
        hf_eligibility[col]
        .value_counts(dropna=False)
    )


# Eligibility text audit
criteria_text = (
    hf_eligibility["criteria"]
    .fillna("")
    .astype(str)
    .str.strip()
)

criteria_length = criteria_text.str.len()

print("\n\n=== ELIGIBILITY CRITERIA TEXT ===")
print("Trials:", len(criteria_text))
print("Non-empty criteria:", (criteria_length > 0).sum())
print("Empty criteria:", (criteria_length == 0).sum())

print("\nCharacter-length summary:")
print(criteria_length.describe())

print("\nExample eligibility text from 3 trials:")

example_idx = criteria_length.nlargest(3).index

for idx in example_idx:
    print("\n" + "=" * 80)
    print("NCT ID:", hf_eligibility.loc[idx, "nct_id"])
    print("Characters:", criteria_length.loc[idx])
    print("-" * 80)

    # Show only first 1000 characters
    print(criteria_text.loc[idx][:1000])

In [ ]:
# STEP 2C — Audit registration timing relative to trial start
# Descriptive only — no modeling.

timing_audit = hf_enrollment_temporal[
    [
        "nct_id",
        "study_first_submitted_date",
        "study_first_posted_date",
        "start_date",
        "submission_year"
    ]
].copy()

# Convert to datetime safely
for col in [
    "study_first_submitted_date",
    "study_first_posted_date",
    "start_date"
]:
    timing_audit[col] = pd.to_datetime(
        timing_audit[col],
        errors="coerce"
    )

# Calculate timing differences
timing_audit["days_submit_to_start"] = (
    timing_audit["start_date"]
    - timing_audit["study_first_submitted_date"]
).dt.days

timing_audit["days_posted_to_start"] = (
    timing_audit["start_date"]
    - timing_audit["study_first_posted_date"]
).dt.days


print("=== DATE AVAILABILITY ===")

for col in [
    "study_first_submitted_date",
    "study_first_posted_date",
    "start_date"
]:
    print(
        f"{col}: "
        f"{timing_audit[col].notna().sum()} / {len(timing_audit)} "
        f"({timing_audit[col].notna().mean()*100:.1f}%)"
    )


valid_submit = timing_audit["days_submit_to_start"].notna()

print("\n=== FIRST SUBMISSION RELATIVE TO TRIAL START ===")

print(
    "Submitted before start:",
    (timing_audit.loc[valid_submit, "days_submit_to_start"] > 0).sum()
)

print(
    "Submitted on start date:",
    (timing_audit.loc[valid_submit, "days_submit_to_start"] == 0).sum()
)

print(
    "Submitted after trial started:",
    (timing_audit.loc[valid_submit, "days_submit_to_start"] < 0).sum()
)

print("\nPercentages:")

timing_categories = pd.Series({
    "Before start":
        (timing_audit.loc[valid_submit, "days_submit_to_start"] > 0).mean() * 100,

    "On start date":
        (timing_audit.loc[valid_submit, "days_submit_to_start"] == 0).mean() * 100,

    "After start":
        (timing_audit.loc[valid_submit, "days_submit_to_start"] < 0).mean() * 100
}).round(1)

print(timing_categories)


print("\n=== TIMING BY SUBMISSION PERIOD ===")

timing_audit["submitted_before_start"] = (
    timing_audit["days_submit_to_start"] > 0
)

timing_audit["period"] = pd.cut(
    timing_audit["submission_year"],
    bins=[1998, 2007, 2012, 2017, 2020, 2024, 2026],
    labels=[
        "1999–2007",
        "2008–2012",
        "2013–2017",
        "2018–2020",
        "2021–2024",
        "2025–2026"
    ]
)

timing_period_summary = (
    timing_audit[
        timing_audit["days_submit_to_start"].notna()
    ]
    .groupby("period", observed=True)
    .agg(
        trials=("nct_id", "count"),
        submitted_before_start=("submitted_before_start", "sum")
    )
    .reset_index()
)

timing_period_summary["before_start_percent"] = (
    timing_period_summary["submitted_before_start"]
    / timing_period_summary["trials"]
    * 100
).round(1)

display(timing_period_summary)

In [ ]:
# STEP 2D — Find existing historical/snapshot-related objects
# Inspection only. Does not modify any data.

keywords = [
    "histor",
    "snapshot",
    "archive",
    "version",
    "registration",
    "submit",
    "posted"
]

matches = []

for name, obj in list(globals().items()):
    name_lower = name.lower()

    if any(keyword in name_lower for keyword in keywords):
        obj_type = type(obj).__name__

        if isinstance(obj, pd.DataFrame):
            description = f"DataFrame {obj.shape}"
        elif isinstance(obj, pd.Series):
            description = f"Series length={len(obj)}"
        else:
            description = obj_type

        matches.append((name, description))

print("=== HISTORICAL / SNAPSHOT RELATED OBJECTS ===\n")

if matches:
    for name, description in sorted(matches):
        print(f"{name:40s} {description}")
else:
    print("No matching objects found.")

In [ ]:
# STEP 2E — Inspect the existing historical_hf_trials object
# No modification or modeling.

print("Shape:", historical_hf_trials.shape)
print("Unique trials:", historical_hf_trials["nct_id"].nunique())

print("\n=== COLUMNS ===")
for col in historical_hf_trials.columns:
    print(col)

print("\n=== FIRST 3 ROWS ===")
display(historical_hf_trials.head(3))

In [ ]:
# STEP 2F — Registration-year coverage of the 2,901 outcome cohort

year_counts = (
    hf_enrollment_temporal
    .groupby("submission_year")
    .agg(
        trials=("nct_id", "count"),
        discontinued=("discontinued", "sum")
    )
    .reset_index()
)

year_counts["completed"] = (
    year_counts["trials"] - year_counts["discontinued"]
)

year_counts["discontinuation_rate"] = (
    year_counts["discontinued"] /
    year_counts["trials"]
).round(3)

display(year_counts)

print("\n=== CUMULATIVE COHORT BY REGISTRATION PERIOD ===")

periods = {
    "Before 2017": hf_enrollment_temporal["submission_year"] < 2017,
    "2017–2020": hf_enrollment_temporal["submission_year"].between(2017, 2020),
    "2021–2024": hf_enrollment_temporal["submission_year"].between(2021, 2024),
    "2025–2026": hf_enrollment_temporal["submission_year"].between(2025, 2026)
}

for label, mask in periods.items():
    subset = hf_enrollment_temporal.loc[mask]

    print(
        f"{label}: "
        f"{len(subset)} trials | "
        f"{subset['discontinued'].sum()} discontinued | "
        f"{subset['discontinued'].mean():.1%} prevalence"
    )

In [ ]:
# STEP 2G — Search for previously created historical snapshot artifacts
# No downloads. No data modification.

import os

search_roots = [
    "/content",
    "/content/drive/MyDrive"
]

keywords = [
    "snapshot",
    "historical",
    "aact",
    "archive",
    "HTT-P1-AACT-HIST",
    "snapshot_strategy"
]

matches = []

for root in search_roots:
    if not os.path.exists(root):
        continue

    for dirpath, dirnames, filenames in os.walk(root):

        # Avoid very large/system folders
        dirnames[:] = [
            d for d in dirnames
            if not d.startswith(".")
        ]

        for filename in filenames:
            full_path = os.path.join(dirpath, filename)
            text = full_path.lower()

            if any(k.lower() in text for k in keywords):
                matches.append(full_path)

print("=== POSSIBLE HISTORICAL ARTIFACTS ===")
print("Files found:", len(matches))

for path in matches[:100]:
    print(path)

if len(matches) > 100:
    print(f"\n... plus {len(matches) - 100} additional matches")

In [ ]:
# STEP 2H — Audit registration timing for potentially reconstructable trials
# Focus: 2017–2024 only. No downloads.

hist_candidate = hf_enrollment_temporal[
    hf_enrollment_temporal["submission_year"].between(2017, 2024)
].copy()

for col in [
    "study_first_submitted_date",
    "study_first_posted_date",
    "start_date"
]:
    hist_candidate[col] = pd.to_datetime(
        hist_candidate[col],
        errors="coerce"
    )

hist_candidate["days_submit_to_post"] = (
    hist_candidate["study_first_posted_date"]
    - hist_candidate["study_first_submitted_date"]
).dt.days

hist_candidate["days_submit_to_start"] = (
    hist_candidate["start_date"]
    - hist_candidate["study_first_submitted_date"]
).dt.days


print("=== 2017–2024 HISTORICAL RECONSTRUCTION CANDIDATES ===")
print("Trials:", len(hist_candidate))
print("Unique NCT IDs:", hist_candidate["nct_id"].nunique())

print("\nSubmission years:")
print(
    hist_candidate["submission_year"]
    .value_counts()
    .sort_index()
)

print("\n=== SUBMISSION → FIRST POSTING DELAY ===")
print(hist_candidate["days_submit_to_post"].describe())

print("\nPosted within:")
for days in [7, 14, 30, 45, 60, 90]:
    n = (
        hist_candidate["days_submit_to_post"]
        .between(0, days)
        .sum()
    )

    print(
        f"{days:>2} days: {n} / {len(hist_candidate)} "
        f"({n / len(hist_candidate):.1%})"
    )

print("\n=== SUBMISSION RELATIVE TO TRIAL START ===")

valid = hist_candidate["days_submit_to_start"].notna()

before = (
    hist_candidate.loc[valid, "days_submit_to_start"] > 0
).sum()

same = (
    hist_candidate.loc[valid, "days_submit_to_start"] == 0
).sum()

after = (
    hist_candidate.loc[valid, "days_submit_to_start"] < 0
).sum()

print("With start date:", valid.sum())
print("Submitted before start:", before)
print("Submitted on start date:", same)
print("Submitted after start:", after)

In [ ]:
# STEP 2I — Build historical archive-matching candidate manifest
# No downloads and no modeling.

archive_candidates = hist_candidate[
    [
        "nct_id",
        "submission_year",
        "study_first_submitted_date",
        "study_first_posted_date",
        "start_date",
        "overall_status",
        "discontinued"
    ]
].copy()

archive_candidates["archive_window_start"] = (
    archive_candidates["study_first_posted_date"]
)

archive_candidates["archive_window_end_45d"] = (
    archive_candidates["study_first_posted_date"]
    + pd.Timedelta(days=45)
)

archive_candidates["submitted_before_start"] = (
    archive_candidates["study_first_submitted_date"]
    < archive_candidates["start_date"]
)

print("=== ARCHIVE-MATCHING MANIFEST ===")
print("Trials:", len(archive_candidates))
print(
    "First-posted range:",
    archive_candidates["study_first_posted_date"].min(),
    "to",
    archive_candidates["study_first_posted_date"].max()
)

print("\nPotential pre-start sensitivity cohort:")
print(
    archive_candidates["submitted_before_start"]
    .value_counts(dropna=False)
)

print("\n=== FIRST 10 CANDIDATES ===")

display(
    archive_candidates[
        [
            "nct_id",
            "submission_year",
            "study_first_posted_date",
            "archive_window_end_45d",
            "submitted_before_start",
            "discontinued"
        ]
    ].head(10)
)

In [ ]:
# STEP 2J — Search notebook variables/code-related strings for AACT archive references
# No downloads.

search_terms = [
    "aact",
    "archive",
    "snapshot",
    "download",
    "clinical_trials"
]

found = []

for name, obj in list(globals().items()):
    try:
        if isinstance(obj, str):
            value = obj

            if any(term in value.lower() for term in search_terms):
                found.append((name, value[:500]))

        elif isinstance(obj, (list, tuple)):
            text = " ".join(map(str, obj))

            if any(term in text.lower() for term in search_terms):
                found.append((name, text[:500]))

    except Exception:
        pass

print("=== POSSIBLE AACT ARCHIVE REFERENCES ===")

if found:
    for name, value in found:
        print(f"\n{name}:")
        print(value)
else:
    print("No AACT archive references found.")

In [ ]:
# STEP 2K — Look for AACT snapshot/archive metadata tables
# Use the existing raw psycopg2 connection instead of SQLAlchemy.

metadata_query = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE
    table_name ILIKE '%%snapshot%%'
    OR table_name ILIKE '%%archive%%'
    OR table_name ILIKE '%%download%%'
ORDER BY table_schema, table_name;
"""

with conn.cursor() as cur:
    cur.execute(metadata_query)
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

snapshot_metadata_tables = pd.DataFrame(
    rows,
    columns=columns
)

print("=== POSSIBLE SNAPSHOT / ARCHIVE METADATA TABLES ===")
print("Rows found:", len(snapshot_metadata_tables))

display(snapshot_metadata_tables)

In [ ]:
# STEP 2L — Match 2017–2020 trials to the next available AACT snapshot
# Uses archive dates from the official AACT historical snapshot listing.
# No snapshot files are downloaded.

archive_dates_2017_2020 = pd.to_datetime([
    # 2017
    "2017-01-05", "2017-02-18", "2017-03-09", "2017-04-03",
    "2017-05-02", "2017-06-13", "2017-07-03", "2017-08-11",
    "2017-09-06", "2017-10-16", "2017-11-02", "2017-12-17",

    # 2018
    "2018-01-11", "2018-02-01", "2018-03-01", "2018-04-01",
    "2018-05-01", "2018-06-01", "2018-07-01", "2018-08-01",
    "2018-09-01", "2018-10-01", "2018-11-01", "2018-12-01",

    # 2019
    "2019-01-01", "2019-02-01", "2019-03-01", "2019-04-01",
    "2019-05-01", "2019-06-01", "2019-07-01", "2019-08-01",
    "2019-09-01", "2019-10-01", "2019-11-01", "2019-12-01",

    # 2020
    "2020-01-01", "2020-02-01", "2020-03-01", "2020-04-01",
    "2020-05-01", "2020-06-01", "2020-07-01", "2020-08-01",
    "2020-09-01", "2020-10-01", "2020-11-01", "2020-12-01"
])

candidate_17_20 = archive_candidates[
    archive_candidates["study_first_posted_date"]
    .between("2017-01-01", "2020-12-31")
].copy()

def next_archive_date(posted_date):
    later = archive_dates_2017_2020[
        archive_dates_2017_2020 >= posted_date
    ]

    if len(later) == 0:
        return pd.NaT

    return later.min()


candidate_17_20["next_archive_date"] = (
    candidate_17_20["study_first_posted_date"]
    .apply(next_archive_date)
)

candidate_17_20["days_posted_to_archive"] = (
    candidate_17_20["next_archive_date"]
    - candidate_17_20["study_first_posted_date"]
).dt.days


print("=== 2017–2020 ARCHIVE COVERAGE ===")
print("Trials:", len(candidate_17_20))
print(
    "Matched to later archive:",
    candidate_17_20["next_archive_date"].notna().sum()
)

print("\nDelay from First Posted to next AACT archive:")
print(candidate_17_20["days_posted_to_archive"].describe())

print("\nCoverage thresholds:")

for days in [30, 45, 60, 90]:
    eligible = (
        candidate_17_20["days_posted_to_archive"]
        .between(0, days)
        .sum()
    )

    print(
        f"≤{days} days: "
        f"{eligible} / {len(candidate_17_20)} "
        f"({eligible / len(candidate_17_20):.1%})"
    )

print("\nBy First Posted year:")

display(
    candidate_17_20
    .assign(
        posted_year=
        candidate_17_20["study_first_posted_date"].dt.year
    )
    .groupby("posted_year")
    .agg(
        trials=("nct_id", "count"),
        median_archive_delay=("days_posted_to_archive", "median"),
        max_archive_delay=("days_posted_to_archive", "max"),
        within_45_days=(
            "days_posted_to_archive",
            lambda x: x.between(0, 45).sum()
        )
    )
    .reset_index()
)

In [ ]:
# STEP 2M — Audit later-period candidates by FIRST POSTED year.
# No downloads and no modeling.

later_candidates = archive_candidates[
    archive_candidates["study_first_posted_date"]
    >= pd.Timestamp("2021-01-01")
].copy()

later_candidates["posted_year"] = (
    later_candidates["study_first_posted_date"].dt.year
)

print("=== LATER ARCHIVE CANDIDATES ===")
print("Trials:", len(later_candidates))

print("\nFirst Posted year:")
print(
    later_candidates["posted_year"]
    .value_counts()
    .sort_index()
)

print("\nFirst Posted date range:")
print(
    later_candidates["study_first_posted_date"].min(),
    "to",
    later_candidates["study_first_posted_date"].max()
)

print("\nSubmission year × First Posted year:")
display(
    pd.crosstab(
        later_candidates["submission_year"],
        later_candidates["posted_year"],
        margins=True
    )
)

In [ ]:
# STEP 2N — Inspect available completion dates for archive candidates
# No downloads and no modeling.

completion_lookup = historical_hf_trials[
    ["nct_id", "completion_date"]
].copy()

completion_lookup["completion_date"] = pd.to_datetime(
    completion_lookup["completion_date"],
    errors="coerce"
)

archive_terminal_check = archive_candidates.merge(
    completion_lookup,
    on="nct_id",
    how="left"
)

archive_terminal_check["days_posted_to_completion"] = (
    archive_terminal_check["completion_date"]
    - archive_terminal_check["study_first_posted_date"]
).dt.days

print("=== TERMINAL-TIMING AUDIT ===")
print("Trials:", len(archive_terminal_check))

print(
    "\nCompletion date available:",
    archive_terminal_check["completion_date"].notna().sum()
)

print("\nDays from First Posted to completion date:")
print(
    archive_terminal_check[
        "days_posted_to_completion"
    ].describe()
)

print("\nTrials with completion date within:")
for days in [0, 30, 45, 60, 90, 180, 365]:
    n = (
        archive_terminal_check["days_posted_to_completion"]
        .between(0, days)
        .sum()
    )
    print(f"≤{days} days: {n}")

print("\nCompletion before First Posted:")
print(
    (
        archive_terminal_check["days_posted_to_completion"] < 0
    ).sum()
)

In [ ]:
# STEP 2O — Audit trials completed before First Posted
# No downloads and no modeling.

archive_terminal_check["completed_before_posting"] = (
    archive_terminal_check["completion_date"]
    < archive_terminal_check["study_first_posted_date"]
)

retro = archive_terminal_check[
    archive_terminal_check["completed_before_posting"]
].copy()

print("=== RETROSPECTIVELY REGISTERED TERMINAL TRIALS ===")
print("Trials:", len(retro))

print("\nBy submission year:")
print(
    retro["submission_year"]
    .value_counts()
    .sort_index()
)

print("\nOutcome:")
print(
    retro["overall_status"]
    .value_counts()
)

print("\nDiscontinued label:")
print(
    retro["discontinued"]
    .value_counts()
)

print("\nSubmitted before trial start:")
print(
    retro["submitted_before_start"]
    .value_counts(dropna=False)
)

print("\nCompletion → First Posted gap (days):")
retro["days_completion_before_posting"] = (
    retro["study_first_posted_date"]
    - retro["completion_date"]
).dt.days

print(
    retro["days_completion_before_posting"]
    .describe()
)

print("\nExamples:")
display(
    retro[
        [
            "nct_id",
            "submission_year",
            "start_date",
            "study_first_posted_date",
            "completion_date",
            "overall_status",
            "discontinued"
        ]
    ].head(15)
)

In [ ]:
# STEP 2O — Classify trials by registration timing relative to completion
# No downloads and no modeling.

timing_audit = archive_terminal_check.copy()

def classify_registration_timing(row):
    posted = row["study_first_posted_date"]
    completed = row["completion_date"]

    if pd.isna(completed):
        return "Completion date missing"

    if completed < posted:
        return "Completed before First Posted"

    if completed == posted:
        return "Completed on First Posted date"

    return "Completed after First Posted"


timing_audit["registration_timing"] = timing_audit.apply(
    classify_registration_timing,
    axis=1
)

summary = (
    timing_audit["registration_timing"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="n")
)

summary["percent"] = (
    summary["n"] / len(timing_audit) * 100
).round(1)

print("=== REGISTRATION-TIMING CLASSIFICATION ===")
print("Total archive candidates:", len(timing_audit))

display(summary)

print("\nBy submission year:")

display(
    pd.crosstab(
        timing_audit["submission_year"],
        timing_audit["registration_timing"],
        margins=True
    )
)

In [ ]:
# STEP 2P — Build exact snapshot manifest for 2017–2020
# No archive downloads.

snapshot_manifest_17_20 = candidate_17_20[
    candidate_17_20["next_archive_date"].notna()
    & candidate_17_20["days_posted_to_archive"].between(0, 45)
].copy()

# Add completion timing
snapshot_manifest_17_20 = snapshot_manifest_17_20.merge(
    completion_lookup,
    on="nct_id",
    how="left"
)

snapshot_manifest_17_20["completion_date"] = pd.to_datetime(
    snapshot_manifest_17_20["completion_date"],
    errors="coerce"
)

# Diagnostic only:
# Was the recorded completion date already reached by the snapshot?
snapshot_manifest_17_20["completion_before_or_on_snapshot"] = (
    snapshot_manifest_17_20["completion_date"]
    <= snapshot_manifest_17_20["next_archive_date"]
)

print("=== 2017–2020 SNAPSHOT MANIFEST ===")

print("Eligible trial-snapshot pairs:",
      len(snapshot_manifest_17_20))

print("Unique snapshots required:",
      snapshot_manifest_17_20["next_archive_date"].nunique())

print("\nTrials by required snapshot:")
display(
    snapshot_manifest_17_20
    .groupby("next_archive_date")
    .agg(
        trials=("nct_id", "count"),
        completion_before_or_on_snapshot=(
            "completion_before_or_on_snapshot", "sum"
        )
    )
    .reset_index()
)

print(
    "\nCompletion date already reached by snapshot:",
    snapshot_manifest_17_20[
        "completion_before_or_on_snapshot"
    ].sum()
)

In [ ]:
# STEP 2Q — Outcome composition after completion-timing screen
# No downloads and no modeling.

manifest_timing = snapshot_manifest_17_20.copy()

manifest_timing["timing_eligible"] = (
    ~manifest_timing["completion_before_or_on_snapshot"]
)

print("=== COMPLETION-TIMING SCREEN ===")

print("Starting trial-snapshot pairs:",
      len(manifest_timing))

print(
    "Completion reached by snapshot:",
    (~manifest_timing["timing_eligible"]).sum()
)

print(
    "Completion still in future:",
    manifest_timing["timing_eligible"].sum()
)

print("\nOutcome among 58 excluded:")
print(
    manifest_timing.loc[
        ~manifest_timing["timing_eligible"],
        "overall_status"
    ].value_counts()
)

print("\nOutcome among 577 remaining:")
print(
    manifest_timing.loc[
        manifest_timing["timing_eligible"],
        "overall_status"
    ].value_counts()
)

print("\nDiscontinuation prevalence among 577 remaining:")
remaining = manifest_timing[
    manifest_timing["timing_eligible"]
]

print(
    remaining["discontinued"].value_counts()
)

print(
    "Prevalence:",
    round(remaining["discontinued"].mean(), 4)
)

In [ ]:
# STEP 2R — Freeze the 2017–2020 reconstruction manifest
# No archive downloads and no modeling.

reconstruction_manifest = manifest_timing[
    manifest_timing["timing_eligible"]
].copy()

reconstruction_manifest = reconstruction_manifest[
    [
        "nct_id",
        "submission_year",
        "study_first_submitted_date",
        "study_first_posted_date",
        "start_date",
        "next_archive_date",
        "days_posted_to_archive",
        "overall_status",
        "discontinued",
        "completion_date"
    ]
].sort_values(
    ["next_archive_date", "nct_id"]
).reset_index(drop=True)

print("=== HISTORICAL RECONSTRUCTION MANIFEST ===")
print("Trials:", len(reconstruction_manifest))
print(
    "Unique snapshots:",
    reconstruction_manifest["next_archive_date"].nunique()
)
print(
    "Snapshot range:",
    reconstruction_manifest["next_archive_date"].min(),
    "to",
    reconstruction_manifest["next_archive_date"].max()
)
print(
    "Unique NCT IDs:",
    reconstruction_manifest["nct_id"].nunique()
)

# Integrity checks based on the reconstructed data.
assert len(reconstruction_manifest) == reconstruction_manifest["nct_id"].nunique()
assert reconstruction_manifest["next_archive_date"].notna().all()
assert reconstruction_manifest["days_posted_to_archive"].between(0, 45).all()

# Reconstructed cohort size and temporal split.
assert len(reconstruction_manifest) == 579

assert (
    reconstruction_manifest["submission_year"] <= 2019
).sum() == 459

assert (
    reconstruction_manifest["submission_year"] == 2020
).sum() == 120

manifest_path = "/content/hearttrialtwin_historical_reconstruction_manifest.csv"

reconstruction_manifest.to_csv(
    manifest_path,
    index=False
)

print("\nIntegrity checks: PASS")
print("Development 2017–2019:", (
    reconstruction_manifest["submission_year"] <= 2019
).sum())
print("Test 2020:", (
    reconstruction_manifest["submission_year"] == 2020
).sum())
print("Total:", len(reconstruction_manifest))
print("Saved:", manifest_path)

In [ ]:
# STEP 2R-A — Audit the 579-trial reconstruction count
# Diagnostic only. Do not modify the manifest.

print("=" * 80)
print("STEP 2R-A — RECONSTRUCTION COUNT AUDIT")
print("=" * 80)

print("\nBasic counts:")
print("Rows:", len(reconstruction_manifest))
print(
    "Unique NCT IDs:",
    reconstruction_manifest["nct_id"].nunique()
)

print("\nSubmission-year distribution:")
print(
    reconstruction_manifest["submission_year"]
    .value_counts()
    .sort_index()
)

print("\nExpected historical split from the existing notebook:")
print("Development 2017–2019: 459")
print("Test 2020: 119")
print("Expected total: 578")

print("\nActual split:")
development_count = (
    reconstruction_manifest["submission_year"] <= 2019
).sum()

test_count = (
    reconstruction_manifest["submission_year"] == 2020
).sum()

print("Development 2017–2019:", development_count)
print("Test 2020:", test_count)
print("Actual total:", development_count + test_count)

print("\nStatus:")
print(
    reconstruction_manifest["overall_status"]
    .value_counts()
)

print("\nDiscontinuation:")
print(
    reconstruction_manifest["discontinued"]
    .value_counts()
)

print("\nDuplicate NCT IDs:")
duplicates = (
    reconstruction_manifest[
        reconstruction_manifest["nct_id"].duplicated(keep=False)
    ]
)

print("Duplicate rows:", len(duplicates))

if len(duplicates) > 0:
    display(
        duplicates.sort_values("nct_id")
    )
else:
    print("None")

print("\nSnapshot distribution:")
display(
    reconstruction_manifest
    .groupby("next_archive_date")
    .size()
    .describe()
)

In [ ]:
# STEP 2S — Audit reconstruction cohort by submission year and outcome
# No downloads and no modeling.

audit = (
    reconstruction_manifest
    .groupby(["submission_year", "overall_status"])
    .size()
    .unstack(fill_value=0)
)

audit["TOTAL"] = audit.sum(axis=1)

print("=== RECONSTRUCTION COHORT BY SUBMISSION YEAR ===")
print(audit)

print("\n=== DISCONTINUATION BY SUBMISSION YEAR ===")

year_outcome = (
    reconstruction_manifest
    .groupby("submission_year")["discontinued"]
    .agg(["count", "sum", "mean"])
    .rename(
        columns={
            "count": "trials",
            "sum": "discontinued_trials",
            "mean": "discontinuation_prevalence"
        }
    )
)

year_outcome["discontinuation_prevalence"] = (
    year_outcome["discontinuation_prevalence"].round(4)
)

print(year_outcome)

print("\nTotal trials:", len(reconstruction_manifest))
print(
    "Overall discontinued:",
    int(reconstruction_manifest["discontinued"].sum())
)
print(
    "Overall prevalence:",
    round(reconstruction_manifest["discontinued"].mean(), 4)
)

In [ ]:
# STEP 2T — Inventory existing historical reconstruction artifacts
# No downloads. No modeling. Read-only.

from pathlib import Path

search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

keywords = (
    "histor",
    "snapshot",
    "aact",
    "reconstruct",
    "hearttrial",
)

matches = []

for root in search_roots:
    if not root.exists():
        continue

    try:
        for p in root.rglob("*"):
            if not p.is_file():
                continue

            name = p.name.lower()

            if any(k in name for k in keywords):
                try:
                    size_mb = p.stat().st_size / (1024 ** 2)
                except OSError:
                    size_mb = None

                matches.append(
                    (str(p), size_mb)
                )
    except (PermissionError, OSError):
        pass

print("=== EXISTING HISTORICAL ARTIFACTS ===")
print("Files found:", len(matches))

for path, size_mb in sorted(matches):
    if size_mb is None:
        print(path)
    else:
        print(f"{size_mb:9.2f} MB | {path}")

In [ ]:
# STEP 2U — Quantify historical snapshot workload by submission year
# No downloads. No modeling.

workload = (
    reconstruction_manifest
    .groupby("submission_year")
    .agg(
        trials=("nct_id", "nunique"),
        snapshots=("next_archive_date", "nunique"),
        discontinued=("discontinued", "sum")
    )
)

workload["completed"] = (
    workload["trials"] - workload["discontinued"]
)

workload["prevalence"] = (
    workload["discontinued"] / workload["trials"]
).round(4)

print("=== HISTORICAL RECONSTRUCTION WORKLOAD ===")
print(workload)

dev = reconstruction_manifest[
    reconstruction_manifest["submission_year"].between(2017, 2019)
]

test = reconstruction_manifest[
    reconstruction_manifest["submission_year"] == 2020
]

print("\n=== PROVISIONAL CHRONOLOGICAL DESIGN ===")

print(
    "Development 2017–2019:",
    len(dev), "trials |",
    dev["next_archive_date"].nunique(), "snapshots"
)

print(
    "Untouched test 2020:",
    len(test), "trials |",
    test["next_archive_date"].nunique(), "snapshots"
)

print(
    "Total unique snapshots required:",
    reconstruction_manifest["next_archive_date"].nunique()
)

In [ ]:
# STEP 2V — Audit snapshot overlap between development and temporal test
# No downloads. No modeling.

dev_snapshot_dates = set(
    reconstruction_manifest.loc[
        reconstruction_manifest["submission_year"].between(2017, 2019),
        "next_archive_date"
    ]
)

test_snapshot_dates = set(
    reconstruction_manifest.loc[
        reconstruction_manifest["submission_year"] == 2020,
        "next_archive_date"
    ]
)

overlap = sorted(dev_snapshot_dates & test_snapshot_dates)

print("=== SNAPSHOT OVERLAP AUDIT ===")
print("Development snapshots:", len(dev_snapshot_dates))
print("2020 test snapshots:", len(test_snapshot_dates))
print("Unique snapshots overall:", len(dev_snapshot_dates | test_snapshot_dates))
print("Snapshots shared by development and test:", len(overlap))

if overlap:
    print("\nShared snapshot dates:")
    for d in overlap:
        print(d)
else:
    print("\nNo snapshot overlap.")

print("\nIMPORTANT:")
print("Chronological separation is by TRIAL submission period, not by archive file.")

In [ ]:
# STEP 2W — Build final 47-snapshot extraction worklist
# No downloads. No modeling.

tmp = reconstruction_manifest.copy()

tmp["cohort_role"] = tmp["submission_year"].apply(
    lambda y: "DEVELOPMENT" if y <= 2019 else "TEST_2020"
)

worklist = (
    tmp.groupby("next_archive_date")
    .agg(
        total_trials=("nct_id", "nunique"),
        development_trials=(
            "cohort_role",
            lambda x: (x == "DEVELOPMENT").sum()
        ),
        test_trials=(
            "cohort_role",
            lambda x: (x == "TEST_2020").sum()
        )
    )
    .reset_index()
    .sort_values("next_archive_date")
)

worklist["role"] = "DEVELOPMENT"

worklist.loc[
    (worklist["development_trials"] > 0) &
    (worklist["test_trials"] > 0),
    "role"
] = "SHARED_ARCHIVE"

worklist.loc[
    (worklist["development_trials"] == 0) &
    (worklist["test_trials"] > 0),
    "role"
] = "TEST_ONLY"

print("=== FINAL HISTORICAL EXTRACTION WORKLIST ===")
print(worklist.to_string(index=False))

print("\n=== SUMMARY ===")
print("Archive files:", len(worklist))
print("Development-only:", (worklist["role"] == "DEVELOPMENT").sum())
print("Shared archives:", (worklist["role"] == "SHARED_ARCHIVE").sum())
print("Test-only:", (worklist["role"] == "TEST_ONLY").sum())

print("\nTrials represented:")
print("Development:", worklist["development_trials"].sum())
print("Test 2020:", worklist["test_trials"].sum())

worklist.to_csv(
    "/content/hearttrialtwin_47_snapshot_worklist.csv",
    index=False
)

print("\nSaved: /content/hearttrialtwin_47_snapshot_worklist.csv")

In [ ]:
# STEP 2X — Freeze NCT-level chronological extraction assignments
# No downloads. No modeling.

final_extraction_manifest = reconstruction_manifest.copy()

final_extraction_manifest["cohort_role"] = (
    final_extraction_manifest["submission_year"]
    .apply(
        lambda y: "DEVELOPMENT_2017_2019"
        if y <= 2019
        else "TEST_2020"
    )
)

final_extraction_manifest = (
    final_extraction_manifest
    .sort_values(["next_archive_date", "nct_id"])
    .reset_index(drop=True)
)

print("=== FROZEN NCT-LEVEL EXTRACTION MANIFEST ===")
print("Trials:", len(final_extraction_manifest))
print(
    "Unique NCT IDs:",
    final_extraction_manifest["nct_id"].nunique()
)
print(
    "Unique snapshots:",
    final_extraction_manifest["next_archive_date"].nunique()
)

print("\nCohort roles:")
print(final_extraction_manifest["cohort_role"].value_counts())

print("\nOutcomes by role:")
print(
    pd.crosstab(
        final_extraction_manifest["cohort_role"],
        final_extraction_manifest["overall_status"]
    )
)

# Integrity checks based on the verified reconstructed cohort.
assert len(final_extraction_manifest) == 579
assert final_extraction_manifest["nct_id"].nunique() == 579

assert (
    final_extraction_manifest["cohort_role"]
    .value_counts()["DEVELOPMENT_2017_2019"] == 459
)

assert (
    final_extraction_manifest["cohort_role"]
    .value_counts()["TEST_2020"] == 120
)

path = "/content/hearttrialtwin_frozen_extraction_manifest.csv"

final_extraction_manifest.to_csv(
    path,
    index=False
)

print("\nIntegrity checks: PASS")
print("Saved:", path)

In [ ]:
# STEP 1 — Inspect candidate structured predictors before modeling

print("hf_design_features:", hf_design_features.shape)
print("Columns:")
print(hf_design_features.columns.tolist())

print("\nhf_lead_sponsors:", hf_lead_sponsors.shape)
print("Columns:")
print(hf_lead_sponsors.columns.tolist())

print("\nhf_geography_features:", hf_geography_features.shape)
print("Columns:")
print(hf_geography_features.columns.tolist())

print("\nhf_eligibility:", hf_eligibility.shape)
print("Columns:")
print(hf_eligibility.columns.tolist())

print("\nhf_enrollment_temporal:", hf_enrollment_temporal.shape)
print("Columns:")
print(hf_enrollment_temporal.columns.tolist())

In [ ]:
# STEP 2 — Build unified HF modeling table
# Keep 2025–2026 out of the primary experiment because outcomes may be immature.

import pandas as pd

base = hf_enrollment_temporal[
    [
        "nct_id",
        "enrollment",
        "enrollment_type",
        "study_first_submitted_date",
        "study_first_posted_date",
        "start_date",
        "overall_status",
        "discontinued",
        "submission_year"
    ]
].copy()

model_df = (
    base
    .merge(hf_design_features, on="nct_id", how="left")
    .merge(
        hf_lead_sponsors[["nct_id", "agency_class"]],
        on="nct_id",
        how="left"
    )
    .merge(hf_geography_features, on="nct_id", how="left")
    .merge(
        hf_eligibility[
            [
                "nct_id",
                "gender",
                "healthy_volunteers",
                "adult",
                "child",
                "older_adult",
                "minimum_age_years",
                "maximum_age_years",
                "criteria"
            ]
        ],
        on="nct_id",
        how="left"
    )
)

# Primary analysis period
model_df = model_df[
    model_df["submission_year"].between(1999, 2024)
].copy()

# Chronological split:
# development = through 2020
# untouched temporal test = 2021–2024
model_df["analysis_split"] = (
    model_df["submission_year"]
    .apply(lambda y: "DEVELOPMENT" if y <= 2020 else "TEMPORAL_TEST")
)

print("Shape:", model_df.shape)
print("Unique NCT IDs:", model_df["nct_id"].nunique())
print("\nSplit:")
print(model_df["analysis_split"].value_counts())

print("\nOutcome by split:")
print(pd.crosstab(
    model_df["analysis_split"],
    model_df["discontinued"],
    margins=True
))

print("\nDiscontinuation prevalence:")
print(
    model_df.groupby("analysis_split")["discontinued"]
    .agg(["count", "sum", "mean"])
)

print("\nSubmission years:")
print(
    model_df.groupby("analysis_split")["submission_year"]
    .agg(["min", "max"])
)

In [ ]:
# STEP 3 — Development-period temporal distribution ONLY
# Do not inspect the 2021–2024 temporal test set.

dev = model_df[
    model_df["analysis_split"] == "DEVELOPMENT"
].copy()

year_summary = (
    dev.groupby("submission_year")["discontinued"]
       .agg(trials="count", discontinued="sum", prevalence="mean")
       .reset_index()
)

print(year_summary.to_string(index=False))

print("\nDevelopment total:", len(dev))
print("Discontinued:", int(dev["discontinued"].sum()))
print("Prevalence:", round(dev["discontinued"].mean(), 4))

In [ ]:
# STEP 4 — Freeze candidate structured predictors and inspect DEVELOPMENT only

structured_features = [
    # Trial design
    "allocation",
    "intervention_model",
    "primary_purpose",
    "masking",
    "subject_masked",
    "caregiver_masked",
    "investigator_masked",
    "outcomes_assessor_masked",

    # Sponsor
    "agency_class",

    # Scale / geography
    "enrollment",
    "number_of_sites",
    "number_of_countries",
    "multinational_trial",

    # Eligibility
    "gender",
    "healthy_volunteers",
    "adult",
    "child",
    "older_adult",
    "minimum_age_years",
    "maximum_age_years"
]

dev = model_df[model_df["analysis_split"] == "DEVELOPMENT"].copy()

audit = []

for col in structured_features:
    audit.append({
        "feature": col,
        "dtype": str(dev[col].dtype),
        "missing_n": int(dev[col].isna().sum()),
        "missing_pct": round(dev[col].isna().mean() * 100, 1),
        "unique_values": int(dev[col].nunique(dropna=True))
    })

audit_df = pd.DataFrame(audit)

print(audit_df.to_string(index=False))

print("\nCandidate structured features:", len(structured_features))
print("Development trials:", len(dev))
print("Temporal test accessed for modeling: NO")

In [ ]:
# STEP 5 — Forward temporal validation: Dummy vs LR vs HGB
# DEVELOPMENT DATA ONLY. 2021–2024 remains untouched.

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# --------------------------------------------------
# Frozen structured predictors
# --------------------------------------------------

categorical_features = [
    "allocation",
    "intervention_model",
    "primary_purpose",
    "masking",
    "agency_class",
    "gender",
    "healthy_volunteers"
]

numeric_features = [
    "enrollment",
    "number_of_sites",
    "number_of_countries",
    "multinational_trial",
    "adult",
    "child",
    "older_adult",
    "minimum_age_years",
    "maximum_age_years"
]

features = categorical_features + numeric_features

# DEVELOPMENT ONLY
dev_model = model_df[
    model_df["analysis_split"] == "DEVELOPMENT"
].copy()

# --------------------------------------------------
# Preprocessing learned separately inside each fold
# --------------------------------------------------

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("cat", categorical_pipe, categorical_features),
    ("num", numeric_pipe, numeric_features)
])

# --------------------------------------------------
# Models
# --------------------------------------------------

models = {
    "Dummy": DummyClassifier(
        strategy="prior"
    ),

    "LogisticRegression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "HistGradientBoosting": Pipeline([
        ("prep", preprocessor),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=200,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        ))
    ])
}

# --------------------------------------------------
# Forward temporal folds
# --------------------------------------------------

folds = [
    ("Fold1", 2014, 2015, 2016),
    ("Fold2", 2016, 2017, 2018),
    ("Fold3", 2018, 2019, 2020)
]

results = []

for fold_name, train_end, val_start, val_end in folds:

    train = dev_model[
        dev_model["submission_year"] <= train_end
    ].copy()

    val = dev_model[
        dev_model["submission_year"].between(
            val_start, val_end
        )
    ].copy()

    X_train = train[features]
    y_train = train["discontinued"].astype(int)

    X_val = val[features]
    y_val = val["discontinued"].astype(int)

    print(
        f"\n{fold_name}: "
        f"train <= {train_end} "
        f"(n={len(train)}, positives={y_train.sum()}) | "
        f"validate {val_start}-{val_end} "
        f"(n={len(val)}, positives={y_val.sum()})"
    )

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        prob = model.predict_proba(X_val)[:, 1]

        prevalence = y_val.mean()

        results.append({
            "fold": fold_name,
            "model": model_name,
            "train_end": train_end,
            "validation_period": f"{val_start}-{val_end}",
            "n_validation": len(val),
            "positives": int(y_val.sum()),
            "prevalence": prevalence,
            "PR_AUC": average_precision_score(y_val, prob),
            "PR_AUC_minus_prevalence":
                average_precision_score(y_val, prob) - prevalence,
            "ROC_AUC": roc_auc_score(y_val, prob),
            "Brier": brier_score_loss(y_val, prob)
        })

results_df = pd.DataFrame(results)

print("\n========== FOLD RESULTS ==========\n")
print(
    results_df.round(4).to_string(index=False)
)

summary = (
    results_df
    .groupby("model")
    .agg(
        mean_PR_AUC=("PR_AUC", "mean"),
        mean_PR_AUC_gain=("PR_AUC_minus_prevalence", "mean"),
        mean_ROC_AUC=("ROC_AUC", "mean"),
        mean_Brier=("Brier", "mean")
    )
    .sort_values("mean_PR_AUC", ascending=False)
)

print("\n========== MEAN FORWARD-VALIDATION PERFORMANCE ==========\n")
print(summary.round(4))

print("\n2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 6 — Leakage diagnostic: what drives HGB?
# DEVELOPMENT ONLY — uses Fold 3 (train <=2018, validate 2019–2020)
# 2021–2024 test remains untouched.

from sklearn.inspection import permutation_importance

train = dev_model[
    dev_model["submission_year"] <= 2018
].copy()

val = dev_model[
    dev_model["submission_year"].between(2019, 2020)
].copy()

X_train = train[features]
y_train = train["discontinued"].astype(int)

X_val = val[features]
y_val = val["discontinued"].astype(int)

hgb = models["HistGradientBoosting"]
hgb.fit(X_train, y_train)

perm = permutation_importance(
    hgb,
    X_val,
    y_val,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = (
    pd.DataFrame({
        "feature": features,
        "importance_mean": perm.importances_mean,
        "importance_sd": perm.importances_std
    })
    .sort_values("importance_mean", ascending=False)
)

print("Fold 3 HGB PR-AUC:",
      round(average_precision_score(
          y_val,
          hgb.predict_proba(X_val)[:, 1]
      ), 4))

print("\nPermutation importance (PR-AUC decrease):")
print(importance_df.round(4).to_string(index=False))

print("\n2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# Check whether enrollment is ANTICIPATED or ACTUAL
# DEVELOPMENT ONLY

print(
    dev_model["enrollment_type"]
    .value_counts(dropna=False)
)

print("\nBy outcome:")
print(
    pd.crosstab(
        dev_model["enrollment_type"],
        dev_model["discontinued"],
        margins=True
    )
)

print("\nMedian enrollment by type and outcome:")
print(
    dev_model.groupby(
        ["enrollment_type", "discontinued"]
    )["enrollment"]
    .agg(["count", "median", "mean"])
    .round(1)
)

In [ ]:
# STEP 7 — Leakage-controlled structured experiment
# Remove current-record enrollment because it overwhelmingly drives
# performance and may represent post-registration ACTUAL enrollment.
# 2021–2024 temporal test remains untouched.

features_no_enrollment = [
    f for f in features if f != "enrollment"
]

numeric_no_enrollment = [
    f for f in numeric_features if f != "enrollment"
]

preprocessor_no_enrollment = ColumnTransformer([
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    ),
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(
                strategy="median",
                add_indicator=True
            )),
            ("scaler", StandardScaler())
        ]),
        numeric_no_enrollment
    )
])

models_no_enrollment = {
    "Dummy": DummyClassifier(strategy="prior"),

    "LogisticRegression": Pipeline([
        ("prep", preprocessor_no_enrollment),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "HistGradientBoosting": Pipeline([
        ("prep", preprocessor_no_enrollment),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=200,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        ))
    ])
}

results_clean = []

for fold_name, train_end, val_start, val_end in folds:

    train = dev_model[
        dev_model["submission_year"] <= train_end
    ].copy()

    val = dev_model[
        dev_model["submission_year"].between(val_start, val_end)
    ].copy()

    X_train = train[features_no_enrollment]
    y_train = train["discontinued"].astype(int)

    X_val = val[features_no_enrollment]
    y_val = val["discontinued"].astype(int)

    for model_name, model in models_no_enrollment.items():

        model.fit(X_train, y_train)
        prob = model.predict_proba(X_val)[:, 1]

        pr = average_precision_score(y_val, prob)

        results_clean.append({
            "fold": fold_name,
            "model": model_name,
            "validation_period": f"{val_start}-{val_end}",
            "n": len(val),
            "positives": int(y_val.sum()),
            "prevalence": y_val.mean(),
            "PR_AUC": pr,
            "PR_AUC_minus_prevalence": pr - y_val.mean(),
            "ROC_AUC": roc_auc_score(y_val, prob),
            "Brier": brier_score_loss(y_val, prob)
        })

results_clean_df = pd.DataFrame(results_clean)

print("===== LEAKAGE-CONTROLLED FOLD RESULTS =====\n")
print(results_clean_df.round(4).to_string(index=False))

clean_summary = (
    results_clean_df
    .groupby("model")
    .agg(
        mean_PR_AUC=("PR_AUC", "mean"),
        mean_PR_AUC_gain=("PR_AUC_minus_prevalence", "mean"),
        mean_ROC_AUC=("ROC_AUC", "mean"),
        mean_Brier=("Brier", "mean")
    )
    .sort_values("mean_PR_AUC", ascending=False)
)

print("\n===== MEAN PERFORMANCE =====\n")
print(clean_summary.round(4))

print("\nEnrollment included: NO")
print("2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 8 — Eligibility-text model
# TF-IDF + Logistic Regression
# Same forward temporal folds; DEVELOPMENT ONLY.
# 2021–2024 test remains untouched.

from sklearn.feature_extraction.text import TfidfVectorizer

text_results = []

for fold_name, train_end, val_start, val_end in folds:

    train = dev_model[
        dev_model["submission_year"] <= train_end
    ].copy()

    val = dev_model[
        dev_model["submission_year"].between(val_start, val_end)
    ].copy()

    train_text = train["criteria"].fillna("")
    val_text = val["criteria"].fillna("")

    y_train = train["discontinued"].astype(int)
    y_val = val["discontinued"].astype(int)

    text_model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.95,
                max_features=10000,
                sublinear_tf=True
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    text_model.fit(train_text, y_train)

    prob = text_model.predict_proba(val_text)[:, 1]

    pr = average_precision_score(y_val, prob)

    text_results.append({
        "fold": fold_name,
        "model": "TFIDF_LogisticRegression",
        "validation_period": f"{val_start}-{val_end}",
        "n": len(val),
        "positives": int(y_val.sum()),
        "prevalence": y_val.mean(),
        "PR_AUC": pr,
        "PR_AUC_minus_prevalence": pr - y_val.mean(),
        "ROC_AUC": roc_auc_score(y_val, prob),
        "Brier": brier_score_loss(y_val, prob)
    })

text_results_df = pd.DataFrame(text_results)

print("===== TEXT-ONLY TEMPORAL VALIDATION =====\n")
print(text_results_df.round(4).to_string(index=False))

print("\n===== MEAN TEXT PERFORMANCE =====\n")
print(
    text_results_df[
        ["PR_AUC", "PR_AUC_minus_prevalence", "ROC_AUC", "Brier"]
    ].mean().round(4)
)

print("\nStructured LR mean PR-AUC:",
      round(
          results_clean_df[
              results_clean_df["model"] == "LogisticRegression"
          ]["PR_AUC"].mean(), 4
      ))

print("2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 9 — Structured + eligibility-text model
# DEVELOPMENT ONLY. 2021–2024 test remains untouched.

combined_results = []

# Separate preprocessing branch for criteria text
combined_preprocessor = ColumnTransformer([
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    ),
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(
                strategy="median",
                add_indicator=True
            )),
            ("scaler", StandardScaler())
        ]),
        numeric_no_enrollment
    ),
    (
        "text",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=10000,
            sublinear_tf=True
        ),
        "criteria"
    )
])

for fold_name, train_end, val_start, val_end in folds:

    train = dev_model[
        dev_model["submission_year"] <= train_end
    ].copy()

    val = dev_model[
        dev_model["submission_year"].between(val_start, val_end)
    ].copy()

    # ColumnTransformer text branch cannot receive None/NaN
    train["criteria"] = train["criteria"].fillna("")
    val["criteria"] = val["criteria"].fillna("")

    combined_features = features_no_enrollment + ["criteria"]

    X_train = train[combined_features]
    y_train = train["discontinued"].astype(int)

    X_val = val[combined_features]
    y_val = val["discontinued"].astype(int)

    combined_model = Pipeline([
        ("prep", combined_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    combined_model.fit(X_train, y_train)

    prob = combined_model.predict_proba(X_val)[:, 1]

    pr = average_precision_score(y_val, prob)

    combined_results.append({
        "fold": fold_name,
        "model": "Structured+TFIDF_LR",
        "validation_period": f"{val_start}-{val_end}",
        "n": len(val),
        "positives": int(y_val.sum()),
        "prevalence": y_val.mean(),
        "PR_AUC": pr,
        "PR_AUC_minus_prevalence": pr - y_val.mean(),
        "ROC_AUC": roc_auc_score(y_val, prob),
        "Brier": brier_score_loss(y_val, prob)
    })

combined_results_df = pd.DataFrame(combined_results)

print("===== COMBINED TEMPORAL VALIDATION =====\n")
print(combined_results_df.round(4).to_string(index=False))

print("\n===== MEAN COMBINED PERFORMANCE =====\n")
print(
    combined_results_df[
        ["PR_AUC", "PR_AUC_minus_prevalence", "ROC_AUC", "Brier"]
    ].mean().round(4)
)

print("\n===== PR-AUC ABLATION =====")
print(
    "Structured LR:",
    round(
        results_clean_df[
            results_clean_df["model"] == "LogisticRegression"
        ]["PR_AUC"].mean(), 4
    )
)

print(
    "Text only:",
    round(text_results_df["PR_AUC"].mean(), 4)
)

print(
    "Structured + text:",
    round(combined_results_df["PR_AUC"].mean(), 4)
)

print("\n2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 10 — Weighted vs unweighted combined LR
# DEVELOPMENT ONLY — same frozen forward temporal folds.
# 2021–2024 test remains untouched.

weight_results = []

for weighting in ["balanced", "unweighted"]:

    for fold_name, train_end, val_start, val_end in folds:

        train = dev_model[
            dev_model["submission_year"] <= train_end
        ].copy()

        val = dev_model[
            dev_model["submission_year"].between(val_start, val_end)
        ].copy()

        train["criteria"] = train["criteria"].fillna("")
        val["criteria"] = val["criteria"].fillna("")

        combined_features = features_no_enrollment + ["criteria"]

        X_train = train[combined_features]
        y_train = train["discontinued"].astype(int)

        X_val = val[combined_features]
        y_val = val["discontinued"].astype(int)

        class_weight = "balanced" if weighting == "balanced" else None

        model = Pipeline([
            ("prep", combined_preprocessor),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight=class_weight,
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)
        prob = model.predict_proba(X_val)[:, 1]

        weight_results.append({
            "weighting": weighting,
            "fold": fold_name,
            "period": f"{val_start}-{val_end}",
            "prevalence": y_val.mean(),
            "PR_AUC": average_precision_score(y_val, prob),
            "ROC_AUC": roc_auc_score(y_val, prob),
            "Brier": brier_score_loss(y_val, prob),
            "mean_predicted_risk": prob.mean()
        })

weight_results_df = pd.DataFrame(weight_results)

print("===== WEIGHTING RESULTS =====\n")
print(weight_results_df.round(4).to_string(index=False))

print("\n===== MEAN PERFORMANCE =====\n")

weight_summary = (
    weight_results_df
    .groupby("weighting")
    .agg(
        mean_PR_AUC=("PR_AUC", "mean"),
        mean_ROC_AUC=("ROC_AUC", "mean"),
        mean_Brier=("Brier", "mean"),
        mean_predicted_risk=("mean_predicted_risk", "mean"),
        mean_observed_prevalence=("prevalence", "mean")
    )
)

print(weight_summary.round(4))

print("\n2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 10 — Class-imbalance strategy comparison
# Combined structured + TF-IDF LR
# DEVELOPMENT ONLY. Temporal test remains untouched.

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

imbalance_strategies = {
    "Unweighted": None,
    "Moderate_2x": {0: 1, 1: 2},
    "Balanced": "balanced"
}

imbalance_results = []

for strategy_name, class_weight in imbalance_strategies.items():

    for fold_name, train_end, val_start, val_end in folds:

        train = dev_model[
            dev_model["submission_year"] <= train_end
        ].copy()

        val = dev_model[
            dev_model["submission_year"].between(val_start, val_end)
        ].copy()

        train["criteria"] = train["criteria"].fillna("")
        val["criteria"] = val["criteria"].fillna("")

        combined_features = features_no_enrollment + ["criteria"]

        X_train = train[combined_features]
        y_train = train["discontinued"].astype(int)

        X_val = val[combined_features]
        y_val = val["discontinued"].astype(int)

        model = Pipeline([
            ("prep", combined_preprocessor),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight=class_weight,
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)

        prob = model.predict_proba(X_val)[:, 1]

        # 0.5 used here only for strategy comparison.
        # Final decision threshold will be selected separately.
        pred = (prob >= 0.5).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_val, pred, labels=[0, 1]
        ).ravel()

        imbalance_results.append({
            "strategy": strategy_name,
            "fold": fold_name,
            "period": f"{val_start}-{val_end}",
            "prevalence": y_val.mean(),

            "PR_AUC": average_precision_score(y_val, prob),
            "ROC_AUC": roc_auc_score(y_val, prob),
            "Brier": brier_score_loss(y_val, prob),

            "precision_0.5": precision_score(
                y_val, pred, zero_division=0
            ),
            "recall_0.5": recall_score(
                y_val, pred, zero_division=0
            ),
            "specificity_0.5": tn / (tn + fp),
            "F1_0.5": f1_score(
                y_val, pred, zero_division=0
            ),

            "mean_predicted_risk": prob.mean()
        })

imbalance_results_df = pd.DataFrame(imbalance_results)

print("===== CLASS-IMBALANCE RESULTS =====\n")
print(
    imbalance_results_df
    .round(4)
    .to_string(index=False)
)

imbalance_summary = (
    imbalance_results_df
    .groupby("strategy")
    .agg(
        mean_PR_AUC=("PR_AUC", "mean"),
        mean_ROC_AUC=("ROC_AUC", "mean"),
        mean_Brier=("Brier", "mean"),
        mean_precision=("precision_0.5", "mean"),
        mean_recall=("recall_0.5", "mean"),
        mean_specificity=("specificity_0.5", "mean"),
        mean_F1=("F1_0.5", "mean"),
        mean_predicted_risk=("mean_predicted_risk", "mean"),
        observed_prevalence=("prevalence", "mean")
    )
)

print("\n===== MEAN PERFORMANCE =====\n")
print(
    imbalance_summary
    .round(4)
    .to_string()
)

print("\n2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 11 — Select operating threshold using DEVELOPMENT temporal predictions
# Final model family is now frozen:
# Unweighted Structured + TF-IDF Logistic Regression
# 2021–2024 temporal test remains untouched.

from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

oof_parts = []

for fold_name, train_end, val_start, val_end in folds:

    train = dev_model[
        dev_model["submission_year"] <= train_end
    ].copy()

    val = dev_model[
        dev_model["submission_year"].between(val_start, val_end)
    ].copy()

    train["criteria"] = train["criteria"].fillna("")
    val["criteria"] = val["criteria"].fillna("")

    combined_features = features_no_enrollment + ["criteria"]

    model = Pipeline([
        ("prep", combined_preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight=None,
            random_state=42
        ))
    ])

    model.fit(
        train[combined_features],
        train["discontinued"].astype(int)
    )

    prob = model.predict_proba(
        val[combined_features]
    )[:, 1]

    oof_parts.append(pd.DataFrame({
        "nct_id": val["nct_id"].values,
        "fold": fold_name,
        "year": val["submission_year"].values,
        "y_true": val["discontinued"].astype(int).values,
        "prob": prob
    }))

oof = pd.concat(oof_parts, ignore_index=True)

# Find threshold maximizing F1 on temporal out-of-fold predictions
precision, recall, thresholds = precision_recall_curve(
    oof["y_true"], oof["prob"]
)

f1_values = (
    2 * precision[:-1] * recall[:-1] /
    (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_values)
selected_threshold = thresholds[best_idx]

pred = (oof["prob"] >= selected_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    oof["y_true"], pred
).ravel()

print("===== FROZEN DEVELOPMENT THRESHOLD =====")
print("Threshold:", round(selected_threshold, 4))
print("Trials:", len(oof))
print("Prevalence:", round(oof["y_true"].mean(), 4))

print("\nOOF discrimination:")
print("PR-AUC:",
      round(average_precision_score(oof["y_true"], oof["prob"]), 4))
print("ROC-AUC:",
      round(roc_auc_score(oof["y_true"], oof["prob"]), 4))
print("Brier:",
      round(brier_score_loss(oof["y_true"], oof["prob"]), 4))

print("\nAt selected threshold:")
print("Precision:",
      round(precision_score(oof["y_true"], pred), 4))
print("Recall/Sensitivity:",
      round(recall_score(oof["y_true"], pred), 4))
print("Specificity:",
      round(tn / (tn + fp), 4))
print("F1:",
      round(f1_score(oof["y_true"], pred), 4))

print("\nConfusion matrix:")
print({"TN": tn, "FP": fp, "FN": fn, "TP": tp})

print("\n2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 12 — Calibration assessment
# DEVELOPMENT temporal out-of-fold predictions ONLY.
# 2021–2024 test remains untouched.

from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

y_oof = oof["y_true"].values
p_oof = oof["prob"].values

# Calibration intercept and slope:
# logit(observed outcome) ~ logit(predicted probability)
eps = 1e-6
p_clip = np.clip(p_oof, eps, 1 - eps)
logit_p = np.log(p_clip / (1 - p_clip))

cal_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000
)

cal_model.fit(
    logit_p.reshape(-1, 1),
    y_oof
)

calibration_intercept = cal_model.intercept_[0]
calibration_slope = cal_model.coef_[0][0]

# Expected Calibration Error (10 equal-frequency bins)
cal_df = pd.DataFrame({
    "y": y_oof,
    "prob": p_oof
})

cal_df["bin"] = pd.qcut(
    cal_df["prob"],
    q=10,
    duplicates="drop"
)

bin_summary = (
    cal_df.groupby("bin", observed=True)
    .agg(
        n=("y", "size"),
        mean_predicted=("prob", "mean"),
        observed_rate=("y", "mean")
    )
    .reset_index()
)

bin_summary["abs_gap"] = abs(
    bin_summary["mean_predicted"] -
    bin_summary["observed_rate"]
)

ece = np.average(
    bin_summary["abs_gap"],
    weights=bin_summary["n"]
)

print("===== DEVELOPMENT CALIBRATION =====")
print("Trials:", len(oof))
print("Observed prevalence:",
      round(oof["y_true"].mean(), 4))
print("Mean predicted probability:",
      round(oof["prob"].mean(), 4))

print("\nCalibration intercept:",
      round(calibration_intercept, 4))
print("Calibration slope:",
      round(calibration_slope, 4))
print("ECE:",
      round(ece, 4))
print("Brier:",
      round(brier_score_loss(y_oof, p_oof), 4))

print("\n===== CALIBRATION BINS =====")
print(
    bin_summary[
        ["n", "mean_predicted", "observed_rate", "abs_gap"]
    ].round(4).to_string(index=False)
)

print("\nSelected threshold remains:",
      round(selected_threshold, 4))

print("2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 13 — Forward temporal calibration experiment
# DEVELOPMENT ONLY.
# Fit base model on earlier data, fit Platt calibration on the next period,
# evaluate on the following future period.
# 2021–2024 remains untouched.

from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# Two genuinely forward calibration experiments:
#
# Experiment 1:
# base train <=2014
# calibrate 2015-2016
# evaluate 2017-2018
#
# Experiment 2:
# base train <=2016
# calibrate 2017-2018
# evaluate 2019-2020

calibration_folds = [
    ("CalFold1", 2014, 2015, 2016, 2017, 2018),
    ("CalFold2", 2016, 2017, 2018, 2019, 2020)
]

calibration_results = []

combined_features = features_no_enrollment + ["criteria"]

for (
    name,
    train_end,
    cal_start,
    cal_end,
    eval_start,
    eval_end
) in calibration_folds:

    train = dev_model[
        dev_model["submission_year"] <= train_end
    ].copy()

    cal = dev_model[
        dev_model["submission_year"].between(cal_start, cal_end)
    ].copy()

    evaluation = dev_model[
        dev_model["submission_year"].between(eval_start, eval_end)
    ].copy()

    for df in [train, cal, evaluation]:
        df["criteria"] = df["criteria"].fillna("")

    # Base unweighted combined LR
    base_model = Pipeline([
        ("prep", combined_preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight=None,
            random_state=42
        ))
    ])

    base_model.fit(
        train[combined_features],
        train["discontinued"].astype(int)
    )

    # Probabilities for calibration period
    p_cal = base_model.predict_proba(
        cal[combined_features]
    )[:, 1]

    y_cal = cal["discontinued"].astype(int).values

    # Platt recalibration:
    # outcome ~ logit(base probability)
    eps = 1e-6

    p_cal_clip = np.clip(p_cal, eps, 1 - eps)
    logit_cal = np.log(
        p_cal_clip / (1 - p_cal_clip)
    ).reshape(-1, 1)

    platt = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000
    )

    platt.fit(logit_cal, y_cal)

    # Future evaluation period
    y_eval = evaluation[
        "discontinued"
    ].astype(int).values

    p_raw = base_model.predict_proba(
        evaluation[combined_features]
    )[:, 1]

    p_raw_clip = np.clip(
        p_raw, eps, 1 - eps
    )

    logit_eval = np.log(
        p_raw_clip / (1 - p_raw_clip)
    ).reshape(-1, 1)

    p_platt = platt.predict_proba(
        logit_eval
    )[:, 1]

    for version, prob in [
        ("Raw", p_raw),
        ("Platt", p_platt)
    ]:

        calibration_results.append({
            "fold": name,
            "version": version,
            "evaluation_period":
                f"{eval_start}-{eval_end}",
            "n": len(evaluation),
            "prevalence": y_eval.mean(),
            "mean_predicted": prob.mean(),
            "PR_AUC":
                average_precision_score(
                    y_eval, prob
                ),
            "ROC_AUC":
                roc_auc_score(
                    y_eval, prob
                ),
            "Brier":
                brier_score_loss(
                    y_eval, prob
                )
        })

calibration_results_df = pd.DataFrame(
    calibration_results
)

print("===== FORWARD CALIBRATION EXPERIMENT =====\n")

print(
    calibration_results_df
    .round(4)
    .to_string(index=False)
)

print("\n===== MEAN BY VERSION =====\n")

print(
    calibration_results_df
    .groupby("version")
    .agg(
        mean_PR_AUC=("PR_AUC", "mean"),
        mean_ROC_AUC=("ROC_AUC", "mean"),
        mean_Brier=("Brier", "mean"),
        mean_predicted=("mean_predicted", "mean"),
        observed_prevalence=("prevalence", "mean")
    )
    .round(4)
)

print("\n2021–2024 TEMPORAL TEST ACCESSED: NO")

In [ ]:
# STEP 14 — FINAL LOCKED TEMPORAL TEST
# Model/preprocessing/threshold are frozen.
# This is the ONE evaluation on 2021–2024.

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# -----------------------------
# Frozen development/test split
# -----------------------------
final_train = dev_model.copy()

final_test = model_df[
    model_df["submission_year"].between(2021, 2024)
].copy()

for df in [final_train, final_test]:
    df["criteria"] = df["criteria"].fillna("")

combined_features = features_no_enrollment + ["criteria"]

X_train = final_train[combined_features]
y_train = final_train["discontinued"].astype(int)

X_test = final_test[combined_features]
y_test = final_test["discontinued"].astype(int)

# -----------------------------
# Frozen final model
# -----------------------------
final_model = Pipeline([
    ("prep", combined_preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight=None,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

# -----------------------------
# ONE temporal-test evaluation
# -----------------------------
test_prob = final_model.predict_proba(X_test)[:, 1]

# Frozen development-selected threshold
final_threshold = 0.2129
test_pred = (test_prob >= final_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_pred,
    labels=[0, 1]
).ravel()

# -----------------------------
# Metrics
# -----------------------------
test_pr_auc = average_precision_score(
    y_test, test_prob
)

test_roc_auc = roc_auc_score(
    y_test, test_prob
)

test_brier = brier_score_loss(
    y_test, test_prob
)

test_precision = precision_score(
    y_test, test_pred, zero_division=0
)

test_recall = recall_score(
    y_test, test_pred, zero_division=0
)

test_specificity = tn / (tn + fp)

test_f1 = f1_score(
    y_test, test_pred, zero_division=0
)

print("===== FINAL LOCKED TEMPORAL TEST =====")
print("Period: 2021–2024")
print("N:", len(final_test))
print("Discontinued:", int(y_test.sum()))
print("Completed:", int((y_test == 0).sum()))
print("Prevalence:", round(y_test.mean(), 4))

print("\n===== DISCRIMINATION / CALIBRATION =====")
print("PR-AUC:", round(test_pr_auc, 4))
print(
    "PR-AUC gain over prevalence:",
    round(test_pr_auc - y_test.mean(), 4)
)
print("ROC-AUC:", round(test_roc_auc, 4))
print("Brier:", round(test_brier, 4))
print(
    "Mean predicted risk:",
    round(test_prob.mean(), 4)
)

print(
    "\n===== FROZEN THRESHOLD:",
    final_threshold,
    "====="
)

print("Precision:", round(test_precision, 4))
print("Recall/Sensitivity:", round(test_recall, 4))
print("Specificity:", round(test_specificity, 4))
print("F1:", round(test_f1, 4))

print("\nConfusion matrix:")
print({
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp)
})

print("\nFINAL TEST ACCESSED: YES — DO NOT RETUNE")

In [ ]:
# STEP 15 — Global interpretation of the FROZEN final model
# No refitting. No test-based tuning.
# Extract logistic-regression coefficients and separate
# structured vs eligibility-text features.

import pandas as pd
import numpy as np

# Get fitted preprocessing + LR
prep = final_model.named_steps["prep"]
lr = final_model.named_steps["model"]

# Get transformed feature names
feature_names = prep.get_feature_names_out()

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": lr.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

# Identify feature source from ColumnTransformer prefixes
coef_df["feature_group"] = np.where(
    coef_df["feature"].str.startswith("text__"),
    "Eligibility text",
    "Structured"
)

# Positive coefficient -> associated with higher model score
# Negative coefficient -> associated with lower model score

structured = coef_df[
    coef_df["feature_group"] == "Structured"
].copy()

text = coef_df[
    coef_df["feature_group"] == "Eligibility text"
].copy()


def show_extremes(df, title, n=15):

    positive = (
        df.sort_values("coefficient", ascending=False)
        .head(n)[["feature", "coefficient"]]
    )

    negative = (
        df.sort_values("coefficient", ascending=True)
        .head(n)[["feature", "coefficient"]]
    )

    print(f"\n===== {title}: HIGHER MODEL SCORE =====")
    print(
        positive.round(4).to_string(index=False)
    )

    print(f"\n===== {title}: LOWER MODEL SCORE =====")
    print(
        negative.round(4).to_string(index=False)
    )


print("Total transformed features:", len(coef_df))
print(
    "Structured features:",
    len(structured)
)
print(
    "Eligibility-text features:",
    len(text)
)

show_extremes(
    structured,
    "STRUCTURED FEATURES",
    n=15
)

show_extremes(
    text,
    "ELIGIBILITY TEXT FEATURES",
    n=20
)

print(
    "\nInterpretation reminder:"
    "\nPositive coefficient = associated with higher predicted "
    "discontinuation score."
    "\nNegative coefficient = associated with lower predicted "
    "discontinuation score."
    "\nThese are model associations, NOT causal effects."
)

print(
    "\nFINAL TEMPORAL TEST WAS NOT USED "
    "TO SELECT OR MODIFY FEATURES."
)

In [ ]:
# STEP 16A — Check SHAP availability
# No model fitting. No model changes.

try:
    import shap
    print("SHAP version:", shap.__version__)
    print("SHAP AVAILABLE: YES")
except Exception as e:
    print("SHAP AVAILABLE: NO")
    print("Error:", e)

print("\nFrozen final model:", type(final_model.named_steps["model"]).__name__)
print("Final threshold:", final_threshold)
print("Transformed features:", len(
    final_model.named_steps["prep"].get_feature_names_out()
))

print("\nFINAL MODEL MODIFIED: NO")
print("FINAL TEST RETUNED: NO")

In [ ]:
# STEP 16B — SHAP global interpretation
# FROZEN final model.
# 2021–2024 test used ONLY for post-hoc interpretation.
# NO model selection, tuning, or feature changes.

import shap
import numpy as np
import pandas as pd

prep = final_model.named_steps["prep"]
lr = final_model.named_steps["model"]

# Transform the locked temporal test
X_test_transformed = prep.transform(
    final_test[combined_features]
)

feature_names = prep.get_feature_names_out()

print("Transformed test shape:", X_test_transformed.shape)

# LinearExplainer for frozen logistic regression
explainer = shap.LinearExplainer(
    lr,
    X_test_transformed
)

shap_values = explainer(
    X_test_transformed
)

print("SHAP values shape:", shap_values.values.shape)

# Mean absolute SHAP = global importance
mean_abs_shap = np.abs(
    shap_values.values
).mean(axis=0)

shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_SHAP": mean_abs_shap
})

shap_importance["feature_group"] = np.where(
    shap_importance["feature"].str.startswith("text__"),
    "Eligibility text",
    "Structured"
)

structured_shap = (
    shap_importance[
        shap_importance["feature_group"] == "Structured"
    ]
    .sort_values("mean_abs_SHAP", ascending=False)
)

text_shap = (
    shap_importance[
        shap_importance["feature_group"] == "Eligibility text"
    ]
    .sort_values("mean_abs_SHAP", ascending=False)
)

print("\n===== TOP 20 STRUCTURED FEATURES BY MEAN |SHAP| =====")
print(
    structured_shap[
        ["feature", "mean_abs_SHAP"]
    ]
    .head(20)
    .round(4)
    .to_string(index=False)
)

print("\n===== TOP 20 ELIGIBILITY-TEXT FEATURES BY MEAN |SHAP| =====")
print(
    text_shap[
        ["feature", "mean_abs_SHAP"]
    ]
    .head(20)
    .round(4)
    .to_string(index=False)
)

print("\n===== TOP 25 FEATURES OVERALL =====")
print(
    shap_importance
    .sort_values("mean_abs_SHAP", ascending=False)
    [["feature", "feature_group", "mean_abs_SHAP"]]
    .head(25)
    .round(4)
    .to_string(index=False)
)

print("\nSHAP INTERPRETATION ONLY — NO RETUNING")
print("FINAL MODEL MODIFIED: NO")

In [ ]:
# STEP 16C — Direction + temporal stability of important structured features
# Interpretation only. NO model tuning.

top_structured_features = (
    structured_shap
    .head(15)["feature"]
    .tolist()
)

# Build dataframe of SHAP values for the top structured features
feature_index = {
    name: i for i, name in enumerate(feature_names)
}

rows = []

for feature in top_structured_features:

    idx = feature_index[feature]

    for year_group, mask in [
        ("2021-2022",
         final_test["submission_year"].between(2021, 2022).values),
        ("2023-2024",
         final_test["submission_year"].between(2023, 2024).values),
        ("2021-2024",
         np.ones(len(final_test), dtype=bool))
    ]:

        vals = shap_values.values[mask, idx]

        rows.append({
            "feature": feature,
            "period": year_group,
            "n": int(mask.sum()),
            "mean_SHAP": vals.mean(),
            "mean_abs_SHAP": np.abs(vals).mean(),
            "positive_SHAP_fraction": (vals > 0).mean()
        })

shap_stability = pd.DataFrame(rows)

print("===== SHAP DIRECTION + TEMPORAL STABILITY =====\n")

for feature in top_structured_features:

    print("\n", feature)

    display_df = shap_stability[
        shap_stability["feature"] == feature
    ][
        [
            "period",
            "n",
            "mean_SHAP",
            "mean_abs_SHAP",
            "positive_SHAP_fraction"
        ]
    ]

    print(
        display_df
        .round(4)
        .to_string(index=False)
    )

print(
    "\nInterpretation:"
    "\nmean_SHAP > 0 = pushes model score upward on average."
    "\nmean_SHAP < 0 = pushes model score downward on average."
    "\nmean_abs_SHAP = contribution magnitude."
    "\npositive_SHAP_fraction = fraction of trials where "
    "the feature contribution is positive."
)

print("\nPOST-HOC INTERPRETATION ONLY — NO RETUNING")

In [ ]:
# STEP 17 — Locked temporal-test error analysis
# NO model changes or retuning.

error_df = final_test[
    [
        "nct_id",
        "submission_year",
        "discontinued",
        "primary_purpose",
        "intervention_model",
        "masking",
        "agency_class",
        "number_of_sites",
        "number_of_countries"
    ]
].copy()

error_df["predicted_risk"] = test_prob
error_df["predicted_class"] = test_pred

error_df["error_type"] = np.select(
    [
        (error_df["discontinued"] == 1) &
        (error_df["predicted_class"] == 1),

        (error_df["discontinued"] == 0) &
        (error_df["predicted_class"] == 0),

        (error_df["discontinued"] == 0) &
        (error_df["predicted_class"] == 1),

        (error_df["discontinued"] == 1) &
        (error_df["predicted_class"] == 0)
    ],
    ["TP", "TN", "FP", "FN"],
    default="Unknown"
)

print("===== ERROR COUNTS =====")
print(error_df["error_type"].value_counts())

print("\n===== PREDICTED RISK BY ERROR TYPE =====")
print(
    error_df.groupby("error_type")["predicted_risk"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

for col in [
    "primary_purpose",
    "intervention_model",
    "masking",
    "agency_class"
]:
    print(f"\n===== {col.upper()} BY ERROR TYPE =====")

    table = pd.crosstab(
        error_df["error_type"],
        error_df[col],
        normalize="index"
    ) * 100

    print(
        table.round(1)
        .to_string()
    )

print("\n===== MOST CONFIDENT FALSE POSITIVES =====")
print(
    error_df[
        error_df["error_type"] == "FP"
    ]
    .sort_values("predicted_risk", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\n===== MOST CONFIDENT FALSE NEGATIVES =====")
print(
    error_df[
        error_df["error_type"] == "FN"
    ]
    .sort_values("predicted_risk", ascending=True)
    .head(10)
    .to_string(index=False)
)

print("\nPOST-HOC ERROR ANALYSIS ONLY — NO RETUNING")

In [ ]:
# STEP 18 — SAVE FINAL FROZEN ARTIFACTS
# No model changes.

import os
import json
import joblib

artifact_dir = "/content/hearttrialtwin_final_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

# 1. Frozen model
joblib.dump(
    final_model,
    f"{artifact_dir}/hearttrialtwin_final_model.joblib"
)

# 2. Final temporal-test predictions
test_output = final_test[
    ["nct_id", "submission_year", "discontinued"]
].copy()

test_output["predicted_risk"] = test_prob
test_output["predicted_class"] = test_pred

test_output.to_csv(
    f"{artifact_dir}/final_temporal_test_predictions.csv",
    index=False
)

# 3. SHAP global importance
shap_importance.to_csv(
    f"{artifact_dir}/shap_global_importance.csv",
    index=False
)

# 4. SHAP temporal stability
shap_stability.to_csv(
    f"{artifact_dir}/shap_temporal_stability.csv",
    index=False
)

# 5. Error analysis
error_df.to_csv(
    f"{artifact_dir}/final_test_error_analysis.csv",
    index=False
)

# 6. Development OOF predictions
oof.to_csv(
    f"{artifact_dir}/development_temporal_oof_predictions.csv",
    index=False
)

# 7. Frozen configuration + headline results
final_summary = {
    "model": "Unweighted Logistic Regression",
    "features": "Structured + eligibility criteria TF-IDF",
    "enrollment_predictor_included": False,
    "posthoc_calibration": "None",
    "threshold": 0.2129,

    "development_oof": {
        "n": 972,
        "prevalence": 0.2202,
        "pr_auc": 0.3209,
        "roc_auc": 0.6116,
        "brier": 0.1680
    },

    "temporal_test": {
        "period": "2021-2024",
        "n": 462,
        "discontinued": 82,
        "completed": 380,
        "prevalence": 0.1775,
        "pr_auc": 0.3102,
        "roc_auc": 0.6017,
        "brier": 0.1433,
        "mean_predicted_risk": 0.2001,
        "precision": 0.2283,
        "recall": 0.5122,
        "specificity": 0.6263,
        "f1": 0.3158,
        "TN": 238,
        "FP": 142,
        "FN": 40,
        "TP": 42
    },

    "notes": [
        "2021-2024 temporal test evaluated once after model freeze.",
        "Current-record enrollment excluded after empirical leakage audit.",
        "Balanced and moderate class weighting evaluated on development data only.",
        "Unweighted LR retained.",
        "Platt calibration evaluated prospectively on development folds and not retained.",
        "Threshold selected using development temporal OOF predictions only.",
        "SHAP and error analyses are post-hoc interpretation only."
    ]
}

with open(
    f"{artifact_dir}/final_model_summary.json",
    "w"
) as f:
    json.dump(final_summary, f, indent=2)

print("===== SAVED FINAL ARTIFACTS =====")
for filename in sorted(os.listdir(artifact_dir)):
    print(filename)

print("\nArtifact directory:", artifact_dir)
print("MODEL RETUNED: NO")
print("FINAL TEST RETUNED: NO")

In [ ]:
# STEP 19 — Package and download frozen artifacts

import shutil
from google.colab import files

zip_path = "/content/hearttrialtwin_final_artifacts.zip"

shutil.make_archive(
    "/content/hearttrialtwin_final_artifacts",
    "zip",
    "/content/hearttrialtwin_final_artifacts"
)

print("Created:", zip_path)

files.download(zip_path)

## STEP 19 — Representative Application Case Review (Post-hoc Only)

This section selects **8 representative cases from the already evaluated 2021–2024 locked temporal test** to satisfy the application-testing requirement. It does **not** retrain the model, retune the threshold, or alter the locked-test results.

Cases: 2 true positives, 2 true negatives, 1 false positive, 1 false negative, and the closest cases immediately below and above the frozen 0.2129 threshold.

In [ ]:
# STEP 19 — Select 8 representative locked-test cases
# POST-HOC APPLICATION REVIEW ONLY — NO RETRAINING / NO RETUNING

import numpy as np
import pandas as pd

FROZEN_THRESHOLD = 0.2129

# error_df was created in the post-hoc error-analysis step and contains:
# nct_id, submission_year, discontinued, predicted_risk,
# predicted_class, error_type, and selected design/provenance fields.
case_pool = error_df.copy()

# Distance from the frozen operating threshold.
case_pool["distance_to_threshold"] = (
    case_pool["predicted_risk"] - FROZEN_THRESHOLD
).abs()

# ---- Normal cases: 2 TP + 2 TN ----
# Choose one more confident and one less confident correct example
# to avoid showing only extreme/easy cases.
tp_sorted = case_pool[case_pool["error_type"] == "TP"].sort_values(
    "predicted_risk", ascending=False
)
tn_sorted = case_pool[case_pool["error_type"] == "TN"].sort_values(
    "predicted_risk", ascending=True
)

tp_cases = pd.concat([
    tp_sorted.head(1),
    tp_sorted.tail(1)
]).drop_duplicates("nct_id").head(2)

tn_cases = pd.concat([
    tn_sorted.head(1),
    tn_sorted.tail(1)
]).drop_duplicates("nct_id").head(2)

# ---- Failure cases: 1 FP + 1 FN ----
# Use confident errors to make the failure modes clear.
fp_case = (
    case_pool[case_pool["error_type"] == "FP"]
    .sort_values("predicted_risk", ascending=False)
    .head(1)
)

fn_case = (
    case_pool[case_pool["error_type"] == "FN"]
    .sort_values("predicted_risk", ascending=True)
    .head(1)
)

# ---- Difficult cases: closest immediately below/above threshold ----
below_case = (
    case_pool[case_pool["predicted_risk"] < FROZEN_THRESHOLD]
    .sort_values("predicted_risk", ascending=False)
    .head(1)
)

above_case = (
    case_pool[case_pool["predicted_risk"] >= FROZEN_THRESHOLD]
    .sort_values("predicted_risk", ascending=True)
    .head(1)
)

# Assemble and label.
selected = []

def add_cases(df, case_category):
    temp = df.copy()
    temp["case_category"] = case_category
    selected.append(temp)

add_cases(tp_cases, "Normal — true positive")
add_cases(tn_cases, "Normal — true negative")
add_cases(fp_case, "Failure — false positive")
add_cases(fn_case, "Failure — false negative")
add_cases(below_case, "Difficult — closest below threshold")
add_cases(above_case, "Difficult — closest above threshold")

representative_cases = pd.concat(selected, ignore_index=True)

# If a near-threshold case duplicates one of the six previously selected cases,
# replace it with the next-nearest unused case on the same side of threshold.
used = set()
final_rows = []

for _, row in representative_cases.iterrows():
    nct = row["nct_id"]
    category = row["case_category"]

    if nct not in used:
        final_rows.append(row)
        used.add(nct)
        continue

    if "below threshold" in category:
        candidates = case_pool[
            (case_pool["predicted_risk"] < FROZEN_THRESHOLD) &
            (~case_pool["nct_id"].isin(used))
        ].sort_values("predicted_risk", ascending=False)
    elif "above threshold" in category:
        candidates = case_pool[
            (case_pool["predicted_risk"] >= FROZEN_THRESHOLD) &
            (~case_pool["nct_id"].isin(used))
        ].sort_values("predicted_risk", ascending=True)
    else:
        candidates = case_pool[~case_pool["nct_id"].isin(used)].sort_values(
            "distance_to_threshold"
        )

    if len(candidates):
        replacement = candidates.iloc[0].copy()
        replacement["case_category"] = category
        final_rows.append(replacement)
        used.add(replacement["nct_id"])

representative_cases = pd.DataFrame(final_rows).reset_index(drop=True)

# Human-readable fields for the report/slide.
representative_cases["actual_outcome"] = representative_cases[
    "discontinued"
].map({0: "Completed", 1: "Discontinued"})

representative_cases["model_result"] = np.where(
    representative_cases["predicted_class"] == 1,
    "At/above threshold",
    "Below threshold"
)

representative_cases["score_pct"] = (
    representative_cases["predicted_risk"] * 100
).round(2)

representative_cases["human_review_focus"] = representative_cases[
    "error_type"
].map({
    "TP": "Confirm evidence supporting elevated score",
    "TN": "Confirm evidence supporting lower score",
    "FP": "Inspect why model overestimated risk",
    "FN": "Inspect missing/weak discontinuation signal"
})

# Near-threshold cases need explicit uncertainty review.
representative_cases.loc[
    representative_cases["case_category"].str.contains("Difficult"),
    "human_review_focus"
] = "Inspect evidence carefully; score is near frozen threshold"

review_columns = [
    "case_category",
    "nct_id",
    "submission_year",
    "actual_outcome",
    "score_pct",
    "model_result",
    "error_type",
    "primary_purpose",
    "intervention_model",
    "masking",
    "agency_class",
    "number_of_sites",
    "number_of_countries",
    "human_review_focus"
]

print("===== 8 REPRESENTATIVE APPLICATION CASES =====")
display(representative_cases[review_columns])

print("\nCase mix:")
print(representative_cases["case_category"].value_counts().to_string())

print("\nFrozen threshold:", FROZEN_THRESHOLD)
print("MODEL RETRAINED: NO")
print("THRESHOLD RETUNED: NO")
print("LOCKED TEST RESULTS CHANGED: NO")
print("POST-HOC APPLICATION REVIEW ONLY")

# Save for Slide 7 / September 30 report.
representative_cases[review_columns].to_csv(
    f"{artifact_dir}/representative_8_case_review.csv",
    index=False
)

print(
    "\nSaved:",
    f"{artifact_dir}/representative_8_case_review.csv"
)


# CHALLENGE 3 — HUMAN DESIGN FOR SEMANTIC MODELING AND GRAPH RAG

## STEP 20 — Human Semantic Inventory

This section begins the Challenge 3 semantic-design work using the actual HeartTrialTwin project data and structures already created in this notebook.

The purpose is to identify the domain entities, attributes, and relationships that are relevant to HeartTrialTwin before implementing a knowledge graph or Graph RAG system.

This section is descriptive only. It does not modify the frozen model, locked future test, threshold, or Challenge 2 results.

In [ ]:
# ============================================================
# STEP 20A — HUMAN SEMANTIC INVENTORY
# ============================================================
# Descriptive only.
# No model training, model modification, or locked-test changes.

print("=" * 80)
print("STEP 20A — HEARTTRIALTWIN SEMANTIC SOURCE INVENTORY")
print("=" * 80)

semantic_sources = {
    "Trial": {
        "dataframe": "discontinuation_model_cohort",
        "source": "ctgov.studies / HeartTrialTwin discontinuation cohort"
    },
    "Condition": {
        "dataframe": "hf_one_row",
        "source": "ctgov.conditions + ctgov.browse_conditions"
    },
    "Intervention": {
        "dataframe": "hf_interventions",
        "source": "ctgov.interventions"
    },
    "Sponsor": {
        "dataframe": "hf_lead_sponsors",
        "source": "ctgov.sponsors"
    },
    "Trial Design": {
        "dataframe": "hf_design_features",
        "source": "ctgov.designs"
    },
    "Eligibility": {
        "dataframe": "hf_eligibility",
        "source": "ctgov.eligibilities"
    },
    "Location": {
        "dataframe": "hf_facilities",
        "source": "ctgov.facilities"
    },
    "Outcome": {
        "dataframe": "hf_design_outcomes",
        "source": "ctgov.design_outcomes"
    }
}

inventory_rows = []

for entity, info in semantic_sources.items():

    df_name = info["dataframe"]

    if df_name in globals():
        df = globals()[df_name]

        inventory_rows.append({
            "Entity": entity,
            "Project dataframe": df_name,
            "Source": info["source"],
            "Rows": len(df),
            "Unique NCT IDs": (
                df["nct_id"].nunique()
                if "nct_id" in df.columns
                else "N/A"
            ),
            "Number of attributes": len(df.columns),
            "Attributes": ", ".join(df.columns.tolist())
        })

    else:
        inventory_rows.append({
            "Entity": entity,
            "Project dataframe": df_name,
            "Source": info["source"],
            "Rows": "NOT AVAILABLE",
            "Unique NCT IDs": "NOT AVAILABLE",
            "Number of attributes": "NOT AVAILABLE",
            "Attributes": "Dataframe not found in current notebook state"
        })

semantic_inventory_df = pd.DataFrame(inventory_rows)

display(semantic_inventory_df)

In [ ]:
# ============================================================
# STEP 20A — CONDITION DATA CHECK
# ============================================================

print("=" * 80)
print("CHECKING ACTUAL CONDITION-LEVEL DATA")
print("=" * 80)

# Show variables in the notebook that look related to conditions
condition_candidates = [
    name for name in globals()
    if any(term in name.lower() for term in [
        "condition",
        "mesh",
        "browse"
    ])
]

print("Potential condition-related objects:")
for name in sorted(condition_candidates):
    obj = globals()[name]
    print(f"\n{name}")
    print(f"Type: {type(obj)}")

    if hasattr(obj, "shape"):
        print(f"Shape: {obj.shape}")

    if hasattr(obj, "columns"):
        print("Columns:")
        print(list(obj.columns))

In [ ]:
# ============================================================
# STEP 20B — ACTUAL HEARTTRIALTWIN ENTITY ATTRIBUTES
# ============================================================

print("=" * 80)
print("STEP 20B — ACTUAL HEARTTRIALTWIN ENTITY ATTRIBUTES")
print("=" * 80)

entity_sources = {
    "Trial": "discontinuation_model_cohort",
    "Condition": "mesh_only_trials",
    "Intervention": "hf_interventions",
    "Sponsor": "hf_lead_sponsors",
    "Trial Design": "hf_design_features",
    "Eligibility": "hf_eligibility",
    "Location": "hf_facilities",
    "Outcome": "hf_design_outcomes"
}

for entity, df_name in entity_sources.items():

    print("\n" + "-" * 80)
    print(f"{entity}  →  {df_name}")

    if df_name not in globals():
        print("Dataframe not found.")
        continue

    df = globals()[df_name]

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    print("\nActual attributes:")
    for col in df.columns:
        print(f"  • {col}")

In [ ]:
# ============================================================
# STEP 20C — PRELIMINARY HEARTTRIALTWIN SEMANTIC SCHEMA
# ============================================================
# Human Design stage only.
# This does NOT create Neo4j nodes or relationships.

print("=" * 80)
print("STEP 20C — PRELIMINARY HEARTTRIALTWIN SEMANTIC SCHEMA")
print("=" * 80)

semantic_schema = pd.DataFrame([
    {
        "Entity": "Trial",
        "Identifier": "nct_id",
        "Source": "discontinuation_model_cohort",
        "Key properties": "brief_title; overall_status; phase; enrollment; start_date; completion_date; brief_summary; hf_subtype; discontinued",
        "Cardinality": "One record per trial",
        "Role": "Central entity in HeartTrialTwin"
    },
    {
        "Entity": "Condition",
        "Identifier": "condition_names / mesh_terms",
        "Source": "mesh_only_trials / condition-related AACT records",
        "Key properties": "condition_names; mesh_terms",
        "Cardinality": "Potentially many conditions per trial",
        "Role": "Represents the clinical condition studied"
    },
    {
        "Entity": "Intervention",
        "Identifier": "nct_id + intervention record",
        "Source": "hf_interventions",
        "Key properties": "intervention_type; intervention_name; description",
        "Cardinality": "Many interventions per trial",
        "Role": "Represents treatments or other interventions"
    },
    {
        "Entity": "Sponsor",
        "Identifier": "lead_sponsor_name",
        "Source": "hf_lead_sponsors",
        "Key properties": "lead_sponsor_name; agency_class",
        "Cardinality": "One lead sponsor per trial in current project data",
        "Role": "Represents trial sponsorship"
    },
    {
        "Entity": "Trial Design",
        "Identifier": "nct_id",
        "Source": "hf_design_features",
        "Key properties": "allocation; intervention_model; primary_purpose; masking; masking-role flags",
        "Cardinality": "Usually one design record per trial",
        "Role": "Represents methodological design"
    },
    {
        "Entity": "Eligibility",
        "Identifier": "nct_id",
        "Source": "hf_eligibility",
        "Key properties": "gender; minimum_age; maximum_age; healthy_volunteers; adult; child; older_adult; criteria",
        "Cardinality": "One eligibility record per trial",
        "Role": "Represents participant eligibility"
    },
    {
        "Entity": "Location",
        "Identifier": "nct_id + site_identifier",
        "Source": "hf_facilities",
        "Key properties": "name; city; state; country; site_identifier",
        "Cardinality": "Many locations per trial",
        "Role": "Represents geographic study sites"
    },
    {
        "Entity": "Outcome",
        "Identifier": "nct_id + outcome record",
        "Source": "hf_design_outcomes",
        "Key properties": "outcome_type; measure; time_frame; description",
        "Cardinality": "Many outcomes per trial",
        "Role": "Represents planned study outcomes"
    },
    {
        "Entity": "Prediction",
        "Identifier": "trial + model/input artifact",
        "Source": "HeartTrialTwin application",
        "Key properties": "experimental score; threshold; threshold status; model identifier; input identifier",
        "Cardinality": "Prediction associated with a selected trial and frozen model/input",
        "Role": "Represents the HeartTrialTwin analytical result"
    },
    {
        "Entity": "Evidence",
        "Identifier": "prediction/evidence record",
        "Source": "HeartTrialTwin application",
        "Key properties": "feature contributions; influential TF-IDF terms; supporting input fields; provenance",
        "Cardinality": "Can support a prediction",
        "Role": "Makes the prediction inspectable"
    },
    {
        "Entity": "Human Review",
        "Identifier": "review record",
        "Source": "HeartTrialTwin application",
        "Key properties": "decision; reviewer note",
        "Cardinality": "One or more reviews may be associated with a prediction",
        "Role": "Preserves human judgment separately from model output"
    }
])

display(semantic_schema)

In [ ]:
# ============================================================
# STEP 20D — PRELIMINARY RELATIONSHIP SCHEMA
# ============================================================

print("=" * 80)
print("STEP 20D — PRELIMINARY RELATIONSHIP SCHEMA")
print("=" * 80)

relationship_schema = pd.DataFrame([
    {
        "Source": "Trial",
        "Relationship": "HAS_CONDITION",
        "Target": "Condition",
        "Why it matters": "Connects a trial to the clinical condition it studies."
    },
    {
        "Source": "Trial",
        "Relationship": "HAS_INTERVENTION",
        "Target": "Intervention",
        "Why it matters": "Connects a trial to its interventions."
    },
    {
        "Source": "Trial",
        "Relationship": "SPONSORED_BY",
        "Target": "Sponsor",
        "Why it matters": "Connects a trial to its lead sponsor."
    },
    {
        "Source": "Trial",
        "Relationship": "HAS_DESIGN",
        "Target": "Trial Design",
        "Why it matters": "Connects a trial to its methodological design."
    },
    {
        "Source": "Trial",
        "Relationship": "HAS_ELIGIBILITY",
        "Target": "Eligibility",
        "Why it matters": "Connects a trial to participant eligibility information."
    },
    {
        "Source": "Trial",
        "Relationship": "HAS_LOCATION",
        "Target": "Location",
        "Why it matters": "Represents the geographic sites associated with a trial."
    },
    {
        "Source": "Trial",
        "Relationship": "HAS_OUTCOME",
        "Target": "Outcome",
        "Why it matters": "Connects a trial to its planned outcomes."
    },
    {
        "Source": "Trial",
        "Relationship": "HAS_PREDICTION",
        "Target": "Prediction",
        "Why it matters": "Connects the trial to the HeartTrialTwin discontinuation-risk result."
    },
    {
        "Source": "Prediction",
        "Relationship": "SUPPORTED_BY",
        "Target": "Evidence",
        "Why it matters": "Connects the model result to inspectable evidence and provenance."
    },
    {
        "Source": "Prediction",
        "Relationship": "HAS_HUMAN_REVIEW",
        "Target": "Human Review",
        "Why it matters": "Preserves reviewer judgment separately from the frozen prediction."
    }
])

display(relationship_schema)

In [ ]:
# ============================================================
# STEP 20E — GRAPH STRUCTURE CHECK
# ============================================================

print("=" * 80)
print("STEP 20E — HEARTTRIALTWIN PRELIMINARY GRAPH")
print("=" * 80)

graph_lines = [
    "Trial",
    "  ├── HAS_CONDITION ────────> Condition",
    "  ├── HAS_INTERVENTION ─────> Intervention",
    "  ├── SPONSORED_BY ─────────> Sponsor",
    "  ├── HAS_DESIGN ───────────> Trial Design",
    "  ├── HAS_ELIGIBILITY ──────> Eligibility",
    "  ├── HAS_LOCATION ─────────> Location",
    "  ├── HAS_OUTCOME ──────────> Outcome",
    "  └── HAS_PREDICTION ───────> Prediction",
    "                                  │",
    "                                  ├── SUPPORTED_BY ──> Evidence",
    "                                  │",
    "                                  └── HAS_HUMAN_REVIEW ──> Human Review"
]

print("\n".join(graph_lines))

# STEP 21 — HUMAN DESIGN: KNOWLEDGE GRAPH QUESTIONS

The purpose of this step is to identify questions that HeartTrialTwin's semantic knowledge graph should help answer.

The questions are selected based on the project's target users, trial data, discontinuation-risk task, evidence requirements, and human-review workflow.

Priority is given to questions that require relationships among multiple entities rather than simple keyword matching.

In [ ]:
# ============================================================
# STEP 21 — KNOWLEDGE GRAPH QUESTIONS
# ============================================================

print("=" * 80)
print("STEP 21 — HEARTTRIALTWIN KNOWLEDGE GRAPH QUESTIONS")
print("=" * 80)

knowledge_graph_questions = pd.DataFrame([
    {
        "Question": "Which heart-failure trials study a particular intervention, and what were their statuses or outcomes?",
        "Relationships required": "Trial → HAS_INTERVENTION → Intervention; Trial → HAS_OUTCOME → Outcome",
        "Why it matters": "Connects intervention context with trial outcomes."
    },
    {
        "Question": "Which trials share the same condition and intervention as a selected trial?",
        "Relationships required": "Trial → HAS_CONDITION → Condition; Trial → HAS_INTERVENTION → Intervention",
        "Why it matters": "Supports identification of related or comparable trials."
    },
    {
        "Question": "What trial-design characteristics are associated with a selected trial and how do they compare with related trials?",
        "Relationships required": "Trial → HAS_DESIGN → Trial Design",
        "Why it matters": "Provides methodological context for trial comparison."
    },
    {
        "Question": "What eligibility characteristics define the population studied by a selected trial?",
        "Relationships required": "Trial → HAS_ELIGIBILITY → Eligibility",
        "Why it matters": "Provides participant-context information when interpreting a trial."
    },
    {
        "Question": "What trial characteristics and evidence are associated with the HeartTrialTwin prediction for a selected trial?",
        "Relationships required": "Trial → HAS_PREDICTION → Prediction → SUPPORTED_BY → Evidence",
        "Why it matters": "Connects the experimental prediction to inspectable evidence."
    },
    {
        "Question": "How does a selected trial relate to other trials through shared conditions, interventions, sponsors, or design characteristics?",
        "Relationships required": "Trial → Condition / Intervention / Sponsor / Trial Design ← related Trial",
        "Why it matters": "Requires multi-hop graph traversal rather than isolated field lookup."
    }
])

display(knowledge_graph_questions)

In [ ]:
# ============================================================
# STEP 22 — PROPOSED GRAPH RAG DESIGN
# ============================================================

print("=" * 80)
print("STEP 22 — HEARTTRIALTWIN GRAPH RAG DESIGN")
print("=" * 80)

graph_rag_components = pd.DataFrame([
    {
        "Stage": "1. User question",
        "Component": "Researcher / user",
        "Purpose": "Ask a clinical-trial question requiring context or relationships."
    },
    {
        "Stage": "2. Entity and concept identification",
        "Component": "Query interpretation",
        "Purpose": "Identify trial IDs, conditions, interventions, sponsors, or other graph concepts."
    },
    {
        "Stage": "3. Graph retrieval",
        "Component": "Neo4j",
        "Purpose": "Retrieve relevant nodes, relationships, and connected trial subgraphs."
    },
    {
        "Stage": "4. Optional text retrieval",
        "Component": "Eligibility / document retrieval",
        "Purpose": "Retrieve detailed text when graph properties alone are insufficient."
    },
    {
        "Stage": "5. Evidence synthesis",
        "Component": "LLM",
        "Purpose": "Synthesize retrieved graph and text evidence into a readable response."
    },
    {
        "Stage": "6. Human review",
        "Component": "Researcher",
        "Purpose": "Inspect evidence and judge whether the response is appropriate."
    }
])

display(graph_rag_components)

In [ ]:
# ============================================================
# STEP 22B — WHAT GRAPH RAG ADDS TO HEARTTRIALTWIN
# ============================================================

print("=" * 80)
print("STEP 22B — WHAT GRAPH RAG ADDS TO HEARTTRIALTWIN")
print("=" * 80)

graph_rag_benefits = pd.DataFrame([
    {
        "Current limitation": "Trial information is distributed across multiple related data structures.",
        "Graph RAG opportunity": "Represent explicit relationships among trials, conditions, interventions, sponsors, designs, eligibility, locations, and outcomes."
    },
    {
        "Current limitation": "Keyword or vector retrieval can find relevant text without explicitly preserving entity relationships.",
        "Graph RAG opportunity": "Retrieve connected subgraphs that preserve relationships among clinical-trial concepts."
    },
    {
        "Current limitation": "Comparing related trials may require multiple joins or separate retrieval steps.",
        "Graph RAG opportunity": "Support multi-hop traversal across shared conditions, interventions, sponsors, designs, and outcomes."
    },
    {
        "Current limitation": "Eligibility criteria contain detailed information that is not fully represented by structured fields.",
        "Graph RAG opportunity": "Combine graph relationships with detailed eligibility-text retrieval when needed."
    },
    {
        "Current limitation": "The current prediction is presented with evidence, but broader trial relationships are not explicitly represented.",
        "Graph RAG opportunity": "Connect predictions to the broader trial context while preserving the frozen model output."
    }
])

display(graph_rag_benefits)

## STEP 22C — Human Design Decision: Prediction–Graph RAG Separation

Graph RAG is proposed as a retrieval and evidence layer around HeartTrialTwin rather than as part of the frozen discontinuation-risk scoring path.

The existing logistic-regression model remains responsible for producing the experimental discontinuation score.

Neo4j and Graph RAG would provide connected trial context, related-trial evidence, and multi-hop retrieval. An LLM could synthesize retrieved evidence for the researcher, but it would not retrain the model, change the frozen score, or replace the deterministic model explanation.

This preserves the separation established in Challenge 2 between model prediction, supporting evidence, and human review.

In [ ]:
# ============================================================
# STEP 22C — PREDICTION / GRAPH RAG BOUNDARY
# ============================================================

print("=" * 80)
print("STEP 22C — HEARTTRIALTWIN PREDICTION / GRAPH RAG BOUNDARY")
print("=" * 80)

architecture_boundary = pd.DataFrame([
    {
        "Layer": "Frozen prediction",
        "Technology": "Final logistic regression",
        "Role": "Produces the experimental discontinuation-risk score.",
        "Can Graph RAG change it?": "No"
    },
    {
        "Layer": "Deterministic model evidence",
        "Technology": "Feature contributions + influential TF-IDF terms",
        "Role": "Shows how available inputs contributed to the frozen prediction.",
        "Can Graph RAG change it?": "No"
    },
    {
        "Layer": "Semantic knowledge",
        "Technology": "Proposed Neo4j knowledge graph",
        "Role": "Represents relationships among trials, conditions, interventions, sponsors, designs, eligibility, locations, and outcomes.",
        "Can Graph RAG change it?": "No"
    },
    {
        "Layer": "Graph retrieval",
        "Technology": "Proposed Graph RAG retrieval",
        "Role": "Retrieves connected trial context and multi-hop evidence.",
        "Can Graph RAG change it?": "No"
    },
    {
        "Layer": "Language synthesis",
        "Technology": "Proposed LLM",
        "Role": "Summarizes retrieved evidence into a researcher-readable response.",
        "Can Graph RAG change it?": "No"
    },
    {
        "Layer": "Human review",
        "Technology": "Researcher",
        "Role": "Evaluates retrieved evidence and the model result.",
        "Can Graph RAG change it?": "No"
    }
])

display(architecture_boundary)

In [ ]:
# ============================================================
# STEP 23A — NEO4J IMPLEMENTATION PLAN: NODE TYPES
# ============================================================

print("=" * 80)
print("STEP 23A — HEARTTRIALTWIN NEO4J NODE TYPES")
print("=" * 80)

neo4j_nodes = pd.DataFrame([
    {
        "Node type": "Trial",
        "Key identifier": "nct_id",
        "Important properties": "title, status, phase, dates, summary, HF subtype"
    },
    {
        "Node type": "Condition",
        "Key identifier": "condition name / MeSH term",
        "Important properties": "condition_names, mesh_terms"
    },
    {
        "Node type": "Intervention",
        "Key identifier": "trial + intervention record",
        "Important properties": "intervention_type, intervention_name, description"
    },
    {
        "Node type": "Sponsor",
        "Key identifier": "sponsor name",
        "Important properties": "lead_sponsor_name, agency_class"
    },
    {
        "Node type": "Trial Design",
        "Key identifier": "trial nct_id",
        "Important properties": "allocation, intervention_model, primary_purpose, masking"
    },
    {
        "Node type": "Eligibility",
        "Key identifier": "trial nct_id",
        "Important properties": "gender, age limits, healthy volunteers, criteria"
    },
    {
        "Node type": "Location",
        "Key identifier": "trial + site_identifier",
        "Important properties": "name, city, state, country"
    },
    {
        "Node type": "Outcome",
        "Key identifier": "trial + outcome record",
        "Important properties": "outcome_type, measure, time_frame, description"
    },
    {
        "Node type": "Prediction",
        "Key identifier": "trial + model/input artifact",
        "Important properties": "score, threshold, threshold status, model identifier"
    },
    {
        "Node type": "Evidence",
        "Key identifier": "prediction/evidence record",
        "Important properties": "feature contributions, TF-IDF terms, provenance"
    },
    {
        "Node type": "Human Review",
        "Key identifier": "review record",
        "Important properties": "decision, reviewer note"
    }
])

display(neo4j_nodes)

In [ ]:
# ============================================================
# STEP 23B — NEO4J IMPLEMENTATION PLAN: RELATIONSHIP TYPES
# ============================================================

print("=" * 80)
print("STEP 23B — HEARTTRIALTWIN NEO4J RELATIONSHIP TYPES")
print("=" * 80)

neo4j_relationships = pd.DataFrame([
    {
        "Source node": "Trial",
        "Relationship": "HAS_CONDITION",
        "Target node": "Condition",
        "Purpose": "Connects a trial to the condition it studies."
    },
    {
        "Source node": "Trial",
        "Relationship": "HAS_INTERVENTION",
        "Target node": "Intervention",
        "Purpose": "Connects a trial to its interventions."
    },
    {
        "Source node": "Trial",
        "Relationship": "SPONSORED_BY",
        "Target node": "Sponsor",
        "Purpose": "Connects a trial to its lead sponsor."
    },
    {
        "Source node": "Trial",
        "Relationship": "HAS_DESIGN",
        "Target node": "Trial Design",
        "Purpose": "Connects a trial to its methodological design."
    },
    {
        "Source node": "Trial",
        "Relationship": "HAS_ELIGIBILITY",
        "Target node": "Eligibility",
        "Purpose": "Connects a trial to participant eligibility information."
    },
    {
        "Source node": "Trial",
        "Relationship": "HAS_LOCATION",
        "Target node": "Location",
        "Purpose": "Connects a trial to its study sites."
    },
    {
        "Source node": "Trial",
        "Relationship": "HAS_OUTCOME",
        "Target node": "Outcome",
        "Purpose": "Connects a trial to its planned outcomes."
    },
    {
        "Source node": "Trial",
        "Relationship": "HAS_PREDICTION",
        "Target node": "Prediction",
        "Purpose": "Connects a selected trial to its HeartTrialTwin prediction."
    },
    {
        "Source node": "Prediction",
        "Relationship": "SUPPORTED_BY",
        "Target node": "Evidence",
        "Purpose": "Connects the prediction to inspectable model/input evidence."
    },
    {
        "Source node": "Prediction",
        "Relationship": "HAS_HUMAN_REVIEW",
        "Target node": "Human Review",
        "Purpose": "Preserves human review separately from the prediction."
    }
])

display(neo4j_relationships)

In [ ]:
# ============================================================
# STEP 23C — NEO4J IMPLEMENTATION PLAN: INITIAL DATA SOURCES
# ============================================================

print("=" * 80)
print("STEP 23C — HEARTTRIALTWIN NEO4J INITIAL DATA SOURCES")
print("=" * 80)

neo4j_data_sources = pd.DataFrame([
    {
        "Graph content": "Trial",
        "Project source": "AACT / ClinicalTrials.gov studies data",
        "Existing project dataframe": "discontinuation_model_cohort",
        "Primary key": "nct_id"
    },
    {
        "Graph content": "Condition",
        "Project source": "AACT conditions and MeSH condition information",
        "Existing project dataframe": "mesh_only_trials / condition-related records",
        "Primary key": "nct_id + condition"
    },
    {
        "Graph content": "Intervention",
        "Project source": "AACT interventions",
        "Existing project dataframe": "hf_interventions",
        "Primary key": "nct_id + intervention record"
    },
    {
        "Graph content": "Sponsor",
        "Project source": "AACT sponsors",
        "Existing project dataframe": "hf_lead_sponsors",
        "Primary key": "nct_id"
    },
    {
        "Graph content": "Trial Design",
        "Project source": "AACT trial-design records",
        "Existing project dataframe": "hf_design_features",
        "Primary key": "nct_id"
    },
    {
        "Graph content": "Eligibility",
        "Project source": "AACT eligibility records",
        "Existing project dataframe": "hf_eligibility",
        "Primary key": "nct_id"
    },
    {
        "Graph content": "Location",
        "Project source": "AACT facility/location records",
        "Existing project dataframe": "hf_facilities",
        "Primary key": "nct_id + site_identifier"
    },
    {
        "Graph content": "Outcome",
        "Project source": "AACT design-outcome records",
        "Existing project dataframe": "hf_design_outcomes",
        "Primary key": "nct_id + outcome record"
    },
    {
        "Graph content": "Prediction / Evidence / Human Review",
        "Project source": "HeartTrialTwin application artifacts",
        "Existing project data": "Frozen model, prediction/evidence artifacts, and review records",
        "Primary key": "trial + model/input/review identifier"
    }
])

display(neo4j_data_sources)

In [ ]:
# ============================================================
# STEP 23D — NEO4J IMPLEMENTATION PLAN: USEFUL GRAPH QUERIES
# ============================================================

print("=" * 80)
print("STEP 23D — HEARTTRIALTWIN USEFUL GRAPH QUERIES")
print("=" * 80)

neo4j_query_design = pd.DataFrame([
    {
        "Query": "Find trials sharing a condition and intervention with a selected trial",
        "Graph pattern": "Trial → HAS_CONDITION → Condition ← HAS_CONDITION ← Related Trial; Trial → HAS_INTERVENTION → Intervention ← HAS_INTERVENTION ← Related Trial",
        "Use": "Identify clinically or methodologically related trials."
    },
    {
        "Query": "Retrieve the prediction and supporting evidence for a selected trial",
        "Graph pattern": "Trial → HAS_PREDICTION → Prediction → SUPPORTED_BY → Evidence",
        "Use": "Provide an evidence-connected view of the HeartTrialTwin prediction without changing the frozen score."
    }
])

display(neo4j_query_design)

In [ ]:
# ============================================================
# STEP 23E — CONSOLIDATED HUMAN DESIGN SUMMARY
# ============================================================

print("=" * 80)
print("STEP 23E — HEARTTRIALTWIN CHALLENGE 3 HUMAN DESIGN SUMMARY")
print("=" * 80)

challenge3_summary = {
    "Core problem":
        "Support human review of heart-failure clinical-trial discontinuation risk "
        "using structured trial data, eligibility evidence, and an experimental model.",

    "Core entity":
        "Trial",

    "Clinical-domain entities":
        [
            "Condition",
            "Intervention",
            "Sponsor",
            "Trial Design",
            "Eligibility",
            "Location",
            "Outcome"
        ],

    "Analytical/review entities":
        [
            "Prediction",
            "Evidence",
            "Human Review"
        ],

    "Core relationships":
        [
            "Trial → HAS_CONDITION → Condition",
            "Trial → HAS_INTERVENTION → Intervention",
            "Trial → SPONSORED_BY → Sponsor",
            "Trial → HAS_DESIGN → Trial Design",
            "Trial → HAS_ELIGIBILITY → Eligibility",
            "Trial → HAS_LOCATION → Location",
            "Trial → HAS_OUTCOME → Outcome",
            "Trial → HAS_PREDICTION → Prediction",
            "Prediction → SUPPORTED_BY → Evidence",
            "Prediction → HAS_HUMAN_REVIEW → Human Review"
        ],

    "Graph RAG role":
        "Retrieve connected trial context and multi-hop evidence for researcher questions.",

    "LLM role":
        "Potentially synthesize retrieved graph and text evidence into a readable response.",

    "Prediction boundary":
        "Graph RAG and the LLM do not modify, retrain, or replace the frozen prediction.",

    "Initial implementation platform":
        "Neo4j",

    "Primary data sources":
        "AACT / ClinicalTrials.gov project data plus HeartTrialTwin prediction and review artifacts."
}

for key, value in challenge3_summary.items():
    print(f"\n{key}:")

    if isinstance(value, list):
        for item in value:
            print(f"  • {item}")
    else:
        print(f"  {value}")

In [ ]:
# ================================================================================
# STEP 24 — HEARTTRIALTWIN RESEARCH-GROUNDED SOURCE INVENTORY
# ================================================================================
# Purpose:
# Document the authoritative/research sources that ground the Knowledge Graph.
# This is research grounding only — no Neo4j implementation yet.
# ================================================================================

research_sources = pd.DataFrame([
    {
        "source": "AACT / ClinicalTrials.gov",
        "source_type": "Dataset / authoritative registry representation",
        "domain": "Clinical trials",
        "key_concepts": (
            "Trial, condition, intervention, sponsor, design, "
            "eligibility, facility/location, outcome, study status"
        ),
        "semantic_concept": (
            "Clinical-trial entities are represented as connected "
            "records across related data structures."
        ),
        "kg_representation": (
            "Trial node connected to Condition, Intervention, Sponsor, "
            "Trial Design, Eligibility, Location, and Outcome nodes."
        ),
        "project_use": (
            "Primary structural foundation because these are the actual "
            "HeartTrialTwin source tables."
        )
    },
    {
        "source": "NLM Medical Subject Headings (MeSH)",
        "source_type": "Controlled vocabulary / semantic standard",
        "domain": "Biomedical terminology",
        "key_concepts": (
            "Controlled biomedical terms, concepts, hierarchical "
            "relationships, standardized terminology"
        ),
        "semantic_concept": (
            "Clinical conditions can be represented using controlled "
            "and hierarchically organized terminology."
        ),
        "kg_representation": (
            "Condition nodes may retain project condition names and "
            "MeSH terms to support normalized clinical semantics."
        ),
        "project_use": (
            "Supports condition normalization and semantic consistency; "
            "not adopted as the complete project ontology."
        )
    },
    {
        "source": (
            "Terminated Trials in the ClinicalTrials.gov Results Database"
        ),
        "source_type": "Peer-reviewed research",
        "domain": "Clinical-trial discontinuation",
        "key_concepts": (
            "Trial termination, reasons for termination, accrual, "
            "trial outcomes, discontinuation context"
        ),
        "semantic_concept": (
            "Trial discontinuation is a meaningful research outcome "
            "with contextual characteristics rather than merely a status label."
        ),
        "kg_representation": (
            "Trial status/discontinuation is retained as a core Trial "
            "property and can be connected to prediction and evidence."
        ),
        "project_use": (
            "Provides research justification for modeling discontinuation "
            "as a central analytical concept."
        )
    }
])

print("=" * 80)
print("STEP 24 — HEARTTRIALTWIN RESEARCH-GROUNDED SOURCE INVENTORY")
print("=" * 80)

display(research_sources)

In [ ]:
# ================================================================================
# STEP 24A — HEARTTRIALTWIN RESEARCH-TO-KNOWLEDGE-GRAPH MAPPING
# ================================================================================
# Purpose:
# Explicitly connect research/authoritative sources to semantic concepts
# and to their proposed representation in the HeartTrialTwin KG.
#
# Required by Challenge 3:
# Research Source → Semantic Concept → Knowledge Graph Representation
# ================================================================================

research_to_kg = pd.DataFrame([
    {
        "research_source": "AACT / ClinicalTrials.gov",
        "semantic_concept": "Clinical trial",
        "research_grounding": (
            "AACT represents ClinicalTrials.gov data as connected "
            "clinical-trial records and related tables."
        ),
        "kg_representation": (
            "Trial node identified by nct_id with properties such as "
            "title, status, phase, enrollment, dates, and summary."
        ),
        "human_design_decision": (
            "Retain Trial as the central domain entity."
        )
    },
    {
        "research_source": "AACT / ClinicalTrials.gov",
        "semantic_concept": (
            "Condition, intervention, sponsor, design, eligibility, "
            "location, and outcome"
        ),
        "research_grounding": (
            "AACT organizes these clinical-trial concepts in related "
            "tables connected through the study identifier."
        ),
        "kg_representation": (
            "Represent them as distinct node types connected to Trial "
            "through explicit relationships."
        ),
        "human_design_decision": (
            "Keep these concepts separate rather than flattening them "
            "into Trial attributes."
        )
    },
    {
        "research_source": "NLM Medical Subject Headings (MeSH)",
        "semantic_concept": "Controlled clinical terminology",
        "research_grounding": (
            "MeSH provides controlled biomedical terminology organized "
            "hierarchically from broader to more specific concepts."
        ),
        "kg_representation": (
            "Condition nodes retain condition_names and MeSH-related "
            "terms so that clinical-condition semantics can be preserved."
        ),
        "human_design_decision": (
            "Use MeSH selectively for condition semantics rather than "
            "adopting the entire MeSH vocabulary."
        )
    },
    {
        "research_source": (
            "Williams et al. (2015), PLOS ONE"
        ),
        "semantic_concept": "Trial discontinuation / termination",
        "research_grounding": (
            "The study examines terminated ClinicalTrials.gov trials "
            "and demonstrates that termination has meaningful research "
            "context and multiple possible reasons."
        ),
        "kg_representation": (
            "Retain overall_status and discontinued as Trial properties "
            "and connect the trial to its HeartTrialTwin Prediction and "
            "Evidence artifacts."
        ),
        "human_design_decision": (
            "Treat discontinuation as a central analytical outcome, "
            "not merely as an incidental status value."
        )
    }
])

print("=" * 80)
print("STEP 24A — HEARTTRIALTWIN RESEARCH → SEMANTIC CONCEPT → KG")
print("=" * 80)

display(research_to_kg)

print("\nResearch-grounding checks:")
print(
    "Number of research mappings:",
    len(research_to_kg)
)

print(
    "Unique sources:",
    research_to_kg["research_source"].nunique()
)

print(
    "Unique semantic concepts:",
    research_to_kg["semantic_concept"].nunique()
)

assert len(research_to_kg) >= 3
assert research_to_kg["research_source"].notna().all()
assert research_to_kg["semantic_concept"].notna().all()
assert research_to_kg["kg_representation"].notna().all()

print("\nResearch → semantic concept → KG mapping: PASS")